# Извлечение атрибутов товаров из поисковых запросов

Портфолио-проект: структурирование коротких запросов маркетплейса в пары  
**«атрибут — значение»**.

## Финальная архитектура

```text
поисковый запрос
       │
       ▼
fastText: предсказание категории товара
       │
       ▼
динамический shortlist из 5 атрибутов
       │
       ▼
базовый multilingual GLiNER2
       │
       ▼
структурированные факты
```

Ноутбук сохраняет полный исследовательский путь:

1. подготовка данных и разрешение конфликтов категорий;
2. обучение fastText-классификатора;
3. построение схем атрибутов категорий;
4. генерация weak-supervised NER-датасета;
5. LoRA-эксперимент;
6. end-to-end инференс и оценка;
7. сравнение полной схемы, shortlist-5 и shortlist-10.

Финальное решение использует **базовый GLiNER2**. LoRA-адаптация на 221 438 автоматически размеченных примерах ухудшила качество и оставлена как отрицательный эксперимент.


## 1. Подготовка окружения

В первой ячейке устанавливаются только библиотеки, необходимые финальному решению. Версия NumPy зафиксирована для совместимости с локальным backend GLiNER2.


In [ ]:
%pip install -q "numpy==1.26.4" "pandas==2.2.2" pyarrow scikit-learn fasttext-wheel "gliner2[local]==1.3.2" tqdm psutil

После выполнения ячейки **рекомендуется перезапустить сеанс**

### Импорты и общие настройки

Все пути и основные параметры собраны в одном месте.


In [ ]:
import gc
import html
import inspect
import json
import os
import pickle
import random
import re
import shutil
import time
import unicodedata
import zipfile
from collections import Counter
from pathlib import Path

import fasttext
import numpy as np
import pandas as pd
import torch

from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm


RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# При запуске из клонированного репозитория PROJECT_ROOT можно не задавать.
PROJECT_ROOT = Path(
    os.environ.get("PROJECT_ROOT", Path.cwd())
).resolve()

DATA_DIR = Path(
    os.environ.get("DATA_DIR", PROJECT_ROOT / "data" / "raw")
)
ARTIFACTS_DIR = Path(
    os.environ.get("ARTIFACTS_DIR", PROJECT_ROOT / "artifacts")
)
RESULTS_DIR = Path(
    os.environ.get("RESULTS_DIR", PROJECT_ROOT / "results")
)

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

QUERY_CLICKS_PATH = DATA_DIR / "query_clicks.parquet"
SKU_DESC_PATH = DATA_DIR / "sku_desc.parquet"
CATALOG_PATH = DATA_DIR / "skus.pkl"

DATASET_PATH = ARTIFACTS_DIR / "classifier_dataset.parquet"
FASTTEXT_TRAIN_PATH = ARTIFACTS_DIR / "fasttext_train.txt"
VALIDATION_PATH = ARTIFACTS_DIR / "fasttext_validation.parquet"
MODEL_PATH = ARTIFACTS_DIR / "subject_classifier_fasttext.bin"
METRICS_PATH = ARTIFACTS_DIR / "subject_classifier_fasttext_metrics.json"

ATTRIBUTE_STATS_PATH = ARTIFACTS_DIR / "category_attribute_stats.pkl"
ATTRIBUTE_CANDIDATES_PATH = ARTIFACTS_DIR / "gliner_attribute_candidates.pkl"
CATEGORY_ATTRIBUTES_PATH = ARTIFACTS_DIR / "category_attributes_by_subject_id.pkl"
CATEGORY_META_PATH = ARTIFACTS_DIR / "category_meta.pkl"

SKU_ATTRIBUTE_VALUES_PATH = ARTIFACTS_DIR / "sku_attribute_values.parquet"
GLINER_WEAK_LABELS_PATH = ARTIFACTS_DIR / "gliner2_weak_labels.parquet"
GLINER_TRAIN_PATH = ARTIFACTS_DIR / "gliner2_train.jsonl"
GLINER_VAL_PATH = ARTIFACTS_DIR / "gliner2_val.jsonl"
GLINER_TEST_PATH = ARTIFACTS_DIR / "gliner2_test.jsonl"
GLINER_DATASET_REPORT_PATH = ARTIFACTS_DIR / "gliner2_dataset_report.json"
GLINER_TRAINING_REPORT_PATH = ARTIFACTS_DIR / "gliner2_training_report.json"
GLINER_COMPARISON_PATH = RESULTS_DIR / "base_vs_lora.csv"

GLINER_BASE_MODEL = "fastino/gliner2-multi-v1"
GLINER_ADAPTER_DIR = ARTIFACTS_DIR / "gliner2_marketplace_lora"
GLINER_ADAPTER_FINAL_DIR = GLINER_ADAPTER_DIR / "final"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Artifacts directory:", ARTIFACTS_DIR)
print("Device:", DEVICE)


## 2. Доступ к данным и артефактам

Публичный репозиторий не содержит исходные данные и обученные веса.

Для полного воспроизведения подготовки данных ожидаются:

```text
data/raw/
├── query_clicks.parquet
├── sku_desc.parquet
└── skus.pkl
```

Для запуска только финального инференса достаточно положить в `artifacts/`:

```text
subject_classifier_fasttext.bin
category_attributes_by_subject_id.pkl
gliner_attribute_candidates.pkl
category_meta.pkl
```

Пути можно переопределить через переменные окружения `DATA_DIR`, `ARTIFACTS_DIR` и `RESULTS_DIR`.


In [ ]:
required_raw_files = [
    QUERY_CLICKS_PATH,
    SKU_DESC_PATH,
    CATALOG_PATH,
]

missing_raw_files = [
    path for path in required_raw_files
    if not path.exists()
]

if missing_raw_files:
    print("Исходные данные не найдены. Подготовительные разделы нужно пропустить:")
    for path in missing_raw_files:
        print(" -", path)
else:
    print("Все исходные файлы найдены.")


## 3. Чтение и первичная проверка таблиц

В выгрузке некоторые названия столбцов могут иметь вид `toValidUTF8(column)`. Они автоматически приводятся к обычному имени. На этом этапе оставляются все исходные столбцы, а обязательная схема проверяется отдельно.


In [ ]:
def normalize_column_name(column_name):
    """Убирает техническую обёртку toValidUTF8(...)."""
    column_name = str(column_name)
    match = re.fullmatch(r"toValidUTF8\((.+)\)", column_name)
    return match.group(1) if match else column_name


query_clicks = pd.read_parquet(QUERY_CLICKS_PATH)
sku_desc = pd.read_parquet(SKU_DESC_PATH)

query_clicks = query_clicks.rename(
    columns={
        column: normalize_column_name(column)
        for column in query_clicks.columns
    }
)

query_clicks = query_clicks.loc[
    :,
    ~query_clicks.columns.duplicated(),
].copy()

required_query_columns = {
    "sku_id",
    "sku_subject_id",
    "query_text",
}

required_card_columns = {
    "sku_id",
    "title",
    "description",
}

missing_query_columns = required_query_columns - set(query_clicks.columns)
missing_card_columns = required_card_columns - set(sku_desc.columns)

if missing_query_columns:
    raise ValueError(
        f"В query_clicks отсутствуют столбцы: "
        f"{sorted(missing_query_columns)}"
    )

if missing_card_columns:
    raise ValueError(
        f"В sku_desc отсутствуют столбцы: "
        f"{sorted(missing_card_columns)}"
    )

print("query_clicks:", query_clicks.shape)
print("sku_desc:", sku_desc.shape)

display(query_clicks.head(3))
display(sku_desc.head(3))


### Конфликты категорий у одного SKU

В логах один товар иногда связан с несколькими `sku_subject_id`. Для обучения классификатора категорий нужен единственный целевой класс, поэтому далее для каждого SKU выбирается наиболее частая категория. При равенстве частот используется меньший ID — только для детерминированности.


In [ ]:
sku_category_diagnostics = (
    query_clicks
    .dropna(subset=["sku_id", "sku_subject_id"])
    .groupby("sku_id")["sku_subject_id"]
    .nunique()
)

print("SKU с известной категорией:", len(sku_category_diagnostics))
print(
    "SKU с несколькими категориями:",
    int((sku_category_diagnostics > 1).sum()),
)
print(
    "Доля конфликтных SKU:",
    f"{(sku_category_diagnostics > 1).mean():.4%}",
)


## 4. Подготовка обучающего датасета для классификатора

Классификатор категории обучается сразу на нескольких типах текста:

- пользовательские запросы;
- названия товаров;
- описания;
- объединённые название и описание.

Карточки одного SKU получают общий `group_id`. Поисковый запрос также образует отдельную группу. Это позволяет разделять train и validation без попадания связанных примеров в обе части.


In [ ]:
MAX_DESCRIPTION_CHARS = 3_000
INCLUDE_FULL_CARD = True

BLOCK_HTML_TAGS = (
    "p|div|br|hr|li|ul|ol|tr|td|th|table|"
    "section|article|header|footer|h[1-6]"
)


def clean_text(value, remove_html=False):
    """Нормализует текст, сохраняя числа, модели и единицы измерения."""
    if value is None or pd.isna(value):
        return ""

    text = unicodedata.normalize("NFKC", str(value))
    text = html.unescape(text)

    if remove_html:
        text = re.sub(
            r"(?is)<(script|style)\b[^>]*>.*?</\1>",
            " ",
            text,
        )
        text = re.sub(
            rf"(?is)</?\s*(?:{BLOCK_HTML_TAGS})\b[^>]*>",
            " ",
            text,
        )
        text = re.sub(r"(?s)<[^>]+>", " ", text)

    text = re.sub(r"[\u200b-\u200d\ufeff]", "", text)
    text = text.lower().replace("ё", "е")
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()


query_clicks_clean = query_clicks.copy()
sku_desc_clean = sku_desc.copy()

query_clicks_clean["sku_id"] = pd.to_numeric(
    query_clicks_clean["sku_id"],
    errors="coerce",
).astype("Int64")

query_clicks_clean["sku_subject_id"] = pd.to_numeric(
    query_clicks_clean["sku_subject_id"],
    errors="coerce",
).astype("Int64")

sku_desc_clean["sku_id"] = pd.to_numeric(
    sku_desc_clean["sku_id"],
    errors="coerce",
).astype("Int64")

query_clicks_clean = query_clicks_clean.dropna(
    subset=["sku_id", "sku_subject_id"]
)

sku_desc_clean = sku_desc_clean.dropna(subset=["sku_id"])


### Каноническая категория товара

Частота связи `SKU → категория` считается по строкам поисковых логов. Полученная таблица далее используется и при подготовке классификатора, и при сопоставлении товаров из каталога с `sku_subject_id`.


In [ ]:
sku_category_counts = (
    query_clicks_clean
    .groupby(["sku_id", "sku_subject_id"], as_index=False)
    .size()
    .rename(columns={"size": "category_count"})
)

sku_category_counts["sku_total_count"] = (
    sku_category_counts
    .groupby("sku_id")["category_count"]
    .transform("sum")
)

sku_category_counts["category_share"] = (
    sku_category_counts["category_count"]
    / sku_category_counts["sku_total_count"]
)

sku_category_counts["n_categories"] = (
    sku_category_counts
    .groupby("sku_id")["sku_subject_id"]
    .transform("nunique")
)

sku_to_category = (
    sku_category_counts
    .sort_values(
        ["sku_id", "category_count", "sku_subject_id"],
        ascending=[True, False, True],
    )
    .drop_duplicates("sku_id")
    .rename(columns={"sku_subject_id": "target"})
    [
        [
            "sku_id",
            "target",
            "category_count",
            "sku_total_count",
            "category_share",
            "n_categories",
        ]
    ]
    .reset_index(drop=True)
)

if not sku_to_category["sku_id"].is_unique:
    raise RuntimeError("После выбора категории sku_id должен быть уникален.")

display(sku_to_category.head())


### Примеры из товарных карточек

Название и описание очищаются отдельно. HTML удаляется только из описаний. Длина описания ограничивается, чтобы длинные карточки не увеличивали время подготовки и размер обучающего файла без заметной пользы.


In [ ]:
cards = (
    sku_desc_clean
    .merge(
        sku_to_category[["sku_id", "target"]],
        on="sku_id",
        how="inner",
        validate="many_to_one",
    )
    .copy()
)

cards["title_clean"] = cards["title"].map(
    lambda value: clean_text(value, remove_html=False)
)

cards["description_clean"] = (
    cards["description"]
    .map(lambda value: clean_text(value, remove_html=True))
    .str.slice(0, MAX_DESCRIPTION_CHARS)
)

title_dataset = (
    cards.loc[
        cards["title_clean"].str.len() > 0,
        ["sku_id", "title_clean", "target"],
    ]
    .rename(columns={"title_clean": "text"})
    .copy()
)

title_dataset["source"] = "title"

description_dataset = (
    cards.loc[
        cards["description_clean"].str.len() > 0,
        ["sku_id", "description_clean", "target"],
    ]
    .rename(columns={"description_clean": "text"})
    .copy()
)

description_dataset["source"] = "description"

card_datasets = [
    title_dataset,
    description_dataset,
]

if INCLUDE_FULL_CARD:
    full_card_mask = (
        (cards["title_clean"].str.len() > 0)
        & (cards["description_clean"].str.len() > 0)
    )

    full_card_dataset = cards.loc[
        full_card_mask,
        [
            "sku_id",
            "title_clean",
            "description_clean",
            "target",
        ],
    ].copy()

    full_card_dataset["text"] = (
        full_card_dataset["title_clean"]
        + " "
        + full_card_dataset["description_clean"]
    ).str.strip()

    full_card_dataset = full_card_dataset[
        ["sku_id", "text", "target"]
    ]

    full_card_dataset["source"] = "full_card"
    card_datasets.append(full_card_dataset)

cards_dataset = pd.concat(card_datasets, ignore_index=True)

cards_dataset["group_id"] = (
    "sku:"
    + cards_dataset["sku_id"].astype(str)
)

cards_dataset = (
    cards_dataset
    .dropna(subset=["sku_id", "text", "target"])
    .loc[lambda frame: frame["text"].str.len() > 0]
    .drop_duplicates(
        subset=["group_id", "text", "target", "source"]
    )
    .reset_index(drop=True)
)

print("Примеров из карточек:", len(cards_dataset))
display(cards_dataset.head())


`target` - то, что будем предсказывать (Категория товаров) для входного текста `text`

### Примеры из поисковых запросов

Один и тот же нормализованный запрос может быть связан с товарами разных категорий. В качестве целевой выбирается наиболее частая категория. Сам запрос становится одной независимой группой.


In [ ]:
query_rows = (
    query_clicks_clean[["sku_id", "query_text"]]
    .merge(
        sku_to_category[["sku_id", "target"]],
        on="sku_id",
        how="inner",
        validate="many_to_one",
    )
    .dropna(subset=["query_text"])
    .copy()
)

query_rows["text"] = query_rows["query_text"].map(
    lambda value: clean_text(value, remove_html=False)
)

query_rows = query_rows.loc[
    query_rows["text"].str.len() > 0
].copy()

query_category_counts = (
    query_rows
    .groupby(["text", "target"], as_index=False)
    .size()
    .rename(columns={"size": "category_count"})
)

query_category_counts["query_total_count"] = (
    query_category_counts
    .groupby("text")["category_count"]
    .transform("sum")
)

query_category_counts["category_share"] = (
    query_category_counts["category_count"]
    / query_category_counts["query_total_count"]
)

query_category_counts["n_categories"] = (
    query_category_counts
    .groupby("text")["target"]
    .transform("nunique")
)

queries_dataset = (
    query_category_counts
    .sort_values(
        ["text", "category_count", "target"],
        ascending=[True, False, True],
    )
    .drop_duplicates("text")
    .reset_index(drop=True)
)

queries_dataset["source"] = "query"

query_group_codes = pd.factorize(
    queries_dataset["text"],
    sort=True,
)[0]

queries_dataset["group_id"] = (
    "query:"
    + pd.Series(query_group_codes, index=queries_dataset.index).astype(str)
)

queries_dataset = queries_dataset[
    [
        "text",
        "target",
        "source",
        "group_id",
        "category_count",
        "query_total_count",
        "category_share",
        "n_categories",
    ]
]

print("Уникальных запросов:", len(queries_dataset))
display(queries_dataset.head())


### Итоговая таблица для fastText

В финальном parquet остаются четыре поля:

- `text` — очищенный текст;
- `target` — `sku_subject_id`;
- `source` — тип исходного текста;
- `group_id` — группа для безопасного train/validation split.


In [ ]:
classifier_dataset = pd.concat(
    [
        cards_dataset[["text", "target", "source", "group_id"]],
        queries_dataset[["text", "target", "source", "group_id"]],
    ],
    ignore_index=True,
)

classifier_dataset["target"] = pd.to_numeric(
    classifier_dataset["target"],
    errors="coerce",
)

classifier_dataset = (
    classifier_dataset
    .dropna(subset=["text", "target", "source", "group_id"])
    .loc[lambda frame: frame["text"].str.len() > 0]
    .drop_duplicates(
        subset=["text", "target", "source", "group_id"]
    )
    .reset_index(drop=True)
)

classifier_dataset["target"] = classifier_dataset["target"].astype(np.int32)
classifier_dataset["group_id"] = classifier_dataset["group_id"].astype(str)

targets_per_group = (
    classifier_dataset[["group_id", "target"]]
    .drop_duplicates()
    .groupby("group_id")["target"]
    .nunique()
)

conflicting_groups = targets_per_group[targets_per_group > 1]

if len(conflicting_groups) > 0:
    raise RuntimeError(
        f"Обнаружено {len(conflicting_groups)} групп "
        "с несколькими целевыми категориями."
    )

classifier_dataset.to_parquet(DATASET_PATH, index=False)

print("Сохранено:", DATASET_PATH)
print("Примеров:", len(classifier_dataset))
print("Групп:", classifier_dataset["group_id"].nunique())
print("Категорий:", classifier_dataset["target"].nunique())
print()
print(classifier_dataset["source"].value_counts())

display(classifier_dataset.head(10))


## 5. Классификатор категории fastText

fastText выбран как компактная модель, устойчиво работающая с опечатками, артикулами и неизвестными формами слов за счёт символьных n-грамм. Плюсом быстро работает. Меньше, чем за миллисекунду.

В GitHub-версии обучение отключено по умолчанию: готовая модель загружается из `artifacts/subject_classifier_fasttext.bin`. Для полного воспроизведения можно включить `RUN_FASTTEXT_TRAINING`.


In [ ]:
VALID_SIZE = 0.20
MIN_GROUPS_PER_CLASS = 5

WRITE_BATCH_SIZE = 10_000
EVAL_BATCH_SIZE = 4_096

SOURCE_REPEAT = {
    "query": 1,
    "title": 3,
    "description": 2,
    "full_card": 1,
}

FASTTEXT_CONFIG = {
    "lr": 0.5,
    "epoch": 20,
    "dim": 128,
    "wordNgrams": 2,
    "minn": 3,
    "maxn": 6,
    "bucket": 2_000_000,
    "minCount": 2,
    "loss": "hs",
    "thread": max(1, os.cpu_count() or 1),
    "verbose": 2,
}


def show_memory(label):
    """Печатает использование оперативной памяти процесса."""
    try:
        import psutil

        memory_gb = psutil.Process().memory_info().rss / 1024**3
        print(f"{label}: {memory_gb:.2f} GB RAM")
    except ImportError:
        pass


def sanitize_fasttext_series(series):
    """Гарантирует, что каждый пример занимает ровно одну строку."""
    return (
        series
        .astype("string")
        .fillna("")
        .str.replace(r"[\r\n\t]+", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


### Фильтрация редких классов и разделение по группам

Категории с менее чем пятью независимыми группами исключаются. Разделение выполняется по `group_id` со стратификацией по категории: связанные тексты одного товара не могут оказаться одновременно в train и validation.


In [ ]:
dataset = pd.read_parquet(
    DATASET_PATH,
    columns=["text", "target", "source", "group_id"],
)

dataset["text"] = dataset["text"].astype("string")
dataset["target"] = pd.to_numeric(
    dataset["target"],
    errors="coerce",
)

dataset = (
    dataset
    .dropna(subset=["text", "target", "source", "group_id"])
    .loc[lambda frame: frame["text"].str.len() > 0]
    .copy()
)

dataset["target"] = dataset["target"].astype(np.int32)
dataset["source"] = dataset["source"].astype("category")
dataset["group_id"] = dataset["group_id"].astype(str)

group_target = (
    dataset[["group_id", "target"]]
    .drop_duplicates()
)

groups_per_class = group_target["target"].value_counts()

valid_classes = groups_per_class[
    groups_per_class >= MIN_GROUPS_PER_CLASS
].index

n_before = len(dataset)

dataset = dataset.loc[
    dataset["target"].isin(valid_classes)
].reset_index(drop=True)

group_table = (
    dataset[["group_id", "target"]]
    .drop_duplicates("group_id")
    .reset_index(drop=True)
)

train_groups, validation_groups = train_test_split(
    group_table,
    test_size=VALID_SIZE,
    random_state=RANDOM_STATE,
    stratify=group_table["target"],
)

train_group_ids = set(train_groups["group_id"])
validation_group_ids = set(validation_groups["group_id"])

if not train_group_ids.isdisjoint(validation_group_ids):
    raise RuntimeError("Train и validation содержат общие группы.")

train_indices = np.flatnonzero(
    dataset["group_id"].isin(train_group_ids).to_numpy()
)

validation_indices = np.flatnonzero(
    dataset["group_id"].isin(validation_group_ids).to_numpy()
)

print("Удалено примеров редких категорий:", n_before - len(dataset))
print("Категорий:", dataset["target"].nunique())
print("Train примеров:", len(train_indices))
print("Validation примеров:", len(validation_indices))
print("Train групп:", len(train_group_ids))
print("Validation групп:", len(validation_group_ids))

show_memory("После подготовки split")


### Формат fastText

Каждая обучающая строка имеет формат:

```text
__label__<sku_subject_id> очищенный текст
```

Validation сохраняется отдельно в parquet, а train записывается потоково, чтобы не создавать крупную промежуточную копию DataFrame в памяти.


In [ ]:
validation_dataset = (
    dataset.iloc[validation_indices]
    [["text", "target", "source", "group_id"]]
    .reset_index(drop=True)
)

validation_dataset.to_parquet(
    VALIDATION_PATH,
    index=False,
)

rng = np.random.default_rng(RANDOM_STATE)
rng.shuffle(train_indices)

n_written = 0
write_started = time.perf_counter()

with open(
    FASTTEXT_TRAIN_PATH,
    "w",
    encoding="utf-8",
    buffering=1024 * 1024,
) as output_file:

    for start in range(0, len(train_indices), WRITE_BATCH_SIZE):
        batch_indices = train_indices[
            start:start + WRITE_BATCH_SIZE
        ]

        batch = dataset.iloc[batch_indices]

        for source_name, repeat_count in SOURCE_REPEAT.items():
            source_mask = (
                batch["source"].astype(str)
                == source_name
            )

            if not source_mask.any():
                continue

            source_batch = batch.loc[
                source_mask,
                ["text", "target"],
            ]

            texts = sanitize_fasttext_series(
                source_batch["text"]
            )

            labels = (
                "__label__"
                + source_batch["target"].astype(str)
            )

            lines = (labels + " " + texts).loc[
                texts.str.len() > 0
            ]

            if len(lines) == 0:
                continue

            payload = "\n".join(lines.tolist()) + "\n"

            for _ in range(repeat_count):
                output_file.write(payload)
                n_written += len(lines)

        processed = min(
            start + WRITE_BATCH_SIZE,
            len(train_indices),
        )

        if (
            start == 0
            or processed == len(train_indices)
            or processed % 200_000 == 0
        ):
            print(
                f"Обработано: {processed:,}/{len(train_indices):,}; "
                f"записано строк: {n_written:,}"
            )

write_seconds = time.perf_counter() - write_started

print(
    f"Train-файл сформирован за "
    f"{write_seconds / 60:.2f} минут."
)
print(
    "Размер train-файла:",
    f"{FASTTEXT_TRAIN_PATH.stat().st_size / 1024**3:.2f} GB",
)


### Обучение или загрузка готовой модели

Стоит заметить, что fastText обучается на CPU. Тип подключённого GPU/TPU на скорость этого этапа практически не влияет.


In [ ]:
RUN_FASTTEXT_TRAINING = False

if RUN_FASTTEXT_TRAINING:
    training_started = time.perf_counter()

    model = fasttext.train_supervised(
        input=str(FASTTEXT_TRAIN_PATH),
        **FASTTEXT_CONFIG,
    )

    training_seconds = time.perf_counter() - training_started
    model.save_model(str(MODEL_PATH))

    print(f"Обучение заняло {training_seconds / 60:.1f} минут")
    print("Модель сохранена:", MODEL_PATH)
else:
    print("Обучение fastText пропущено.")


In [ ]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Не найдена готовая fastText-модель: {MODEL_PATH}. "
        "Положи файл в artifacts/ или включи RUN_FASTTEXT_TRAINING."
    )

model = fasttext.load_model(str(MODEL_PATH))
print("fastText загружен:", MODEL_PATH)


### Оценка классификатора

Рассчитываются:

- `Accuracy@1`;
- `Recall@3`;
- Macro-F1;
- Weighted-F1;
- метрики отдельно для каждого источника текста.


In [ ]:
validation_dataset = pd.read_parquet(VALIDATION_PATH)

y_true = validation_dataset["target"].to_numpy(dtype=np.int32)
y_pred = np.empty(len(validation_dataset), dtype=np.int32)
top3_hits = np.zeros(len(validation_dataset), dtype=bool)

evaluation_started = time.perf_counter()

for start in range(
    0,
    len(validation_dataset),
    EVAL_BATCH_SIZE,
):
    stop = min(
        start + EVAL_BATCH_SIZE,
        len(validation_dataset),
    )

    texts = sanitize_fasttext_series(
        validation_dataset["text"].iloc[start:stop]
    ).tolist()

    predicted_labels, _ = model.predict(texts, k=3)

    for local_index, labels in enumerate(predicted_labels):
        global_index = start + local_index

        predicted_ids = [
            int(label.removeprefix("__label__"))
            for label in labels
        ]

        y_pred[global_index] = predicted_ids[0]
        top3_hits[global_index] = (
            y_true[global_index] in predicted_ids
        )

evaluation_seconds = time.perf_counter() - evaluation_started

metrics = {
    "accuracy_at_1": float(
        accuracy_score(y_true, y_pred)
    ),
    "recall_at_3": float(top3_hits.mean()),
    "macro_f1": float(
        f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        )
    ),
    "weighted_f1": float(
        f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        )
    ),
    "evaluation_seconds": float(evaluation_seconds),
    "n_validation_examples": int(len(validation_dataset)),
}

with open(METRICS_PATH, "w", encoding="utf-8") as file:
    json.dump(
        metrics,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(f"Accuracy@1: {metrics['accuracy_at_1']:.4f}")
print(f"Recall@3: {metrics['recall_at_3']:.4f}")
print(f"Macro-F1: {metrics['macro_f1']:.4f}")
print(f"Weighted-F1: {metrics['weighted_f1']:.4f}")
print(
    f"Оценка заняла "
    f"{evaluation_seconds / 60:.2f} минут."
)


In [ ]:
source_values = (
    validation_dataset["source"]
    .astype(str)
    .to_numpy()
)

source_metrics = []

for source_name in sorted(np.unique(source_values)):
    source_mask = source_values == source_name

    source_metrics.append(
        {
            "source": source_name,
            "n": int(source_mask.sum()),
            "accuracy_at_1": accuracy_score(
                y_true[source_mask],
                y_pred[source_mask],
            ),
            "recall_at_3": float(
                top3_hits[source_mask].mean()
            ),
            "macro_f1": f1_score(
                y_true[source_mask],
                y_pred[source_mask],
                average="macro",
                zero_division=0,
            ),
        }
    )

source_metrics_df = pd.DataFrame(source_metrics)
display(source_metrics_df)


## 6. Построение схем атрибутов по категориям

GLiNER2 не должен выбирать сущность среди всех атрибутов каталога одновременно. Для каждого `sku_subject_id` строится компактный список наиболее характерных атрибутов.

Источник категорий — поисковые логи, а источник атрибутов — товарный каталог. Сопоставление выполняется через `sku_id`.


### Чтение товарного каталога

Каталог имеет XML-подобную структуру после преобразования в pickle. Вспомогательные функции ниже приводят одиночные словари и списки к единому формату.


In [ ]:
try:
    catalog = pd.read_pickle(CATALOG_PATH)
except Exception:
    with open(CATALOG_PATH, "rb") as file:
        catalog = pickle.load(file)

shop = catalog["yml_catalog"]["shop"]


def ensure_list(value):
    """Приводит одиночное значение или список к списку."""
    if value is None:
        return []

    return value if isinstance(value, list) else [value]


def unwrap_container(container, item_key):
    """Извлекает элементы из XML-подобного контейнера."""
    if isinstance(container, dict) and item_key in container:
        return ensure_list(container[item_key])

    return ensure_list(container)


categories = unwrap_container(
    shop["categories"],
    "category",
)

offers = unwrap_container(
    shop["offers"],
    "offer",
)

print("Категорий в каталоге:", len(categories))
print("Товаров в каталоге:", len(offers))


### Нормализация категорий и атрибутов

Поля `vendor` и `model` рассматриваются как наиболее надёжные источники атрибутов **Бренд** и **Модель**. Параметры с суффиксом `_search` имеют повышенный приоритет, а технический суффикс удаляется из финального имени.


In [ ]:
def normalize_id(value):
    """Приводит числовой идентификатор к строке без .0."""
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    normalized = str(value).strip()

    if normalized.endswith(".0") and normalized[:-2].isdigit():
        normalized = normalized[:-2]

    return normalized or None

In [ ]:
def extract_values(value):
    """Рекурсивно извлекает текстовые значения."""
    if value is None:
        return []

    if isinstance(value, list):
        result = []

        for item in value:
            result.extend(extract_values(item))

        return result

    if isinstance(value, dict):
        if "#text" in value:
            return extract_values(value["#text"])

        if "@id" in value:
            return extract_values(value["@id"])

        return []

    text = str(value).strip()
    return [text] if text else []


def normalize_attribute_name(name):
    """Очищает имя атрибута от технических деталей."""
    if name is None:
        return None

    normalized = unicodedata.normalize(
        "NFKC",
        str(name),
    )

    normalized = re.sub(
        r"_search$",
        "",
        normalized.strip(),
        flags=re.IGNORECASE,
    )

    normalized = re.sub(
        r"\s+",
        " ",
        normalized,
    ).strip()

    return normalized or None


category_meta = {}

for category in categories:
    if not isinstance(category, dict):
        continue

    category_id = normalize_id(category.get("@id"))

    if category_id is None:
        continue

    category_meta[category_id] = {
        "category_name": str(
            category.get("#text", category_id)
        ).strip(),
        "parent_id": normalize_id(
            category.get("@parentId")
        ),
    }

print("Категорий с метаданными:", len(category_meta))


### Связь SKU с `sku_subject_id`

Используется та же каноническая категория, которая была определена при подготовке классификатора. В статистику атрибутов входят только категории, присутствующие среди классов обученной fastText-модели.


In [ ]:
sku_to_subject = {
    normalize_id(row.sku_id): normalize_id(row.target)
    for row in sku_to_category.itertuples(index=False)
}

model_category_ids = {
    normalize_id(
        label.removeprefix("__label__")
    )
    for label in model.get_labels()
}

catalog_offer_ids = {
    normalize_id(offer.get("@id"))
    for offer in offers
    if isinstance(offer, dict)
    and offer.get("@id") is not None
}

overlap = catalog_offer_ids & set(sku_to_subject)

print("Классов fastText:", len(model_category_ids))
print("SKU в каталоге:", len(catalog_offer_ids))
print("SKU с категорией в логах:", len(sku_to_subject))
print("Пересечение:", len(overlap))
print(
    "Покрытие каталога:",
    f"{len(overlap) / max(len(catalog_offer_ids), 1):.2%}",
)


### Подсчёт статистик

Для каждой пары «категория — атрибут» считаются:

- число товаров категории;
- число товаров с атрибутом;
- доля покрытия;
- число вхождений из точных источников;
- наличие атрибута в `vendor` или `model`.


In [ ]:
category_offer_count = Counter()
attribute_offer_count = Counter()
attribute_precise_count = Counter()
attribute_core_count = Counter()

for offer in tqdm(
    offers,
    desc="Подсчёт атрибутов по sku_subject_id",
):
    if not isinstance(offer, dict):
        continue

    sku_id = normalize_id(offer.get("@id"))
    subject_id = sku_to_subject.get(sku_id)

    if subject_id is None:
        continue

    if subject_id not in model_category_ids:
        continue

    offer_attributes = {}

    def add_attribute(name, value, priority):
        attribute_name = normalize_attribute_name(name)
        values = extract_values(value)

        if not attribute_name or not values:
            return

        offer_attributes[attribute_name] = max(
            priority,
            offer_attributes.get(attribute_name, 0),
        )

    add_attribute(
        "Бренд",
        offer.get("vendor"),
        priority=3,
    )

    add_attribute(
        "Модель",
        offer.get("model"),
        priority=3,
    )

    for parameter in ensure_list(offer.get("param")):
        if not isinstance(parameter, dict):
            continue

        raw_name = parameter.get("@name")

        if raw_name is None:
            continue

        is_search_attribute = bool(
            re.search(
                r"_search$",
                str(raw_name),
                flags=re.IGNORECASE,
            )
        )

        add_attribute(
            raw_name,
            parameter.get("#text"),
            priority=2 if is_search_attribute else 1,
        )

    category_offer_count[subject_id] += 1

    for attribute_name, priority in offer_attributes.items():
        key = (subject_id, attribute_name)

        attribute_offer_count[key] += 1

        if priority >= 2:
            attribute_precise_count[key] += 1

        if priority == 3:
            attribute_core_count[key] += 1


attribute_rows = []

for (
    category_id,
    attribute_name,
), attribute_n_offers in attribute_offer_count.items():

    category_n_offers = category_offer_count[category_id]
    key = (category_id, attribute_name)

    attribute_rows.append(
        {
            "category_id": category_id,
            "category_name": category_meta.get(
                category_id,
                {},
            ).get("category_name"),
            "category_n_offers": category_n_offers,
            "attribute_name": attribute_name,
            "attribute_n_offers": attribute_n_offers,
            "coverage": (
                attribute_n_offers
                / category_n_offers
            ),
            "precise_n_offers": (
                attribute_precise_count[key]
            ),
            "precise_coverage": (
                attribute_precise_count[key]
                / category_n_offers
            ),
            "core_n_offers": (
                attribute_core_count[key]
            ),
        }
    )

attribute_stats = pd.DataFrame(attribute_rows)

if attribute_stats.empty:
    raise RuntimeError(
        "Не удалось построить статистику атрибутов. "
        "Проверьте пересечение sku_id между источниками."
    )

print(
    "Категорий с товарами:",
    attribute_stats["category_id"].nunique(),
)
print("Пар категория–атрибут:", len(attribute_stats))

display(attribute_stats.head())


### Отбор кандидатов для GLiNER2

Порог зависит от размера категории:

- в маленьких категориях достаточно нескольких товаров;
- в крупных требуется больше абсолютных наблюдений;
- **Бренд** и **Модель** сохраняются, если присутствуют в каталоге;
- на одну категорию передаётся не более 25 атрибутов.

Ранжирование учитывает покрытие, точность источника, статистическую поддержку и принадлежность к основным полям.


In [ ]:
stats = attribute_stats.copy()

stats["normalized_attribute_name"] = (
    stats["attribute_name"]
    .astype(str)
    .str.casefold()
    .str.strip()
)

technical_attributes = {
    "потратить бонусы",
    "продавец",
}

n_offers = stats["category_n_offers"]

stats["min_attribute_offers"] = np.select(
    [
        n_offers <= 2,
        n_offers < 10,
        n_offers < 20,
        n_offers < 100,
    ],
    [
        1,
        2,
        3,
        5,
    ],
    default=10,
)

stats["min_coverage"] = np.select(
    [
        n_offers <= 2,
        n_offers < 10,
        n_offers < 20,
        n_offers < 100,
    ],
    [
        0.50,
        0.20,
        0.10,
        0.05,
    ],
    default=0.03,
)

regular_attribute_mask = (
    (
        stats["attribute_n_offers"]
        >= stats["min_attribute_offers"]
    )
    & (
        stats["coverage"]
        >= stats["min_coverage"]
    )
)

core_attribute_mask = stats["core_n_offers"] > 0

candidate_mask = (
    (regular_attribute_mask | core_attribute_mask)
    & (
        ~stats["normalized_attribute_name"]
        .isin(technical_attributes)
    )
)

attribute_candidates = stats.loc[
    candidate_mask
].copy()

support_score = (
    np.log1p(attribute_candidates["attribute_n_offers"])
    / np.log1p(
        attribute_candidates["category_n_offers"]
        .clip(lower=1)
    )
)

attribute_candidates["is_core"] = (
    attribute_candidates["core_n_offers"] > 0
).astype(int)

attribute_candidates["score"] = (
    0.50 * attribute_candidates["coverage"]
    + 0.20 * attribute_candidates["precise_coverage"]
    + 0.20 * support_score
    + 0.10 * attribute_candidates["is_core"]
)

attribute_candidates["max_attributes"] = np.select(
    [
        attribute_candidates["category_n_offers"] < 5,
        attribute_candidates["category_n_offers"] < 20,
    ],
    [
        12,
        18,
    ],
    default=25,
)

attribute_candidates = attribute_candidates.sort_values(
    [
        "category_id",
        "is_core",
        "score",
        "coverage",
        "attribute_n_offers",
    ],
    ascending=[
        True,
        False,
        False,
        False,
        False,
    ],
)

attribute_candidates["attribute_rank"] = (
    attribute_candidates
    .groupby("category_id")
    .cumcount()
    + 1
)

attribute_candidates = (
    attribute_candidates.loc[
        attribute_candidates["attribute_rank"]
        <= attribute_candidates["max_attributes"]
    ]
    .reset_index(drop=True)
)

category_attributes = (
    attribute_candidates
    .sort_values(["category_id", "attribute_rank"])
    .groupby("category_id")["attribute_name"]
    .agg(
        lambda values: list(dict.fromkeys(values))
    )
    .to_dict()
)

print("Категорий со схемой:", len(category_attributes))
print(
    "Среднее число атрибутов:",
    f"{np.mean([len(values) for values in category_attributes.values()]):.2f}",
)

if "16434" in category_attributes:
    print()
    print("Пример для категории 16434:")
    print(category_attributes["16434"])


### Сохранение схем

Сохраняются как итоговый словарь для инференса, так и подробные таблицы — они полезны для анализа отбора и дальнейшей настройки порогов.


In [ ]:
attribute_stats.to_pickle(ATTRIBUTE_STATS_PATH)
attribute_candidates.to_pickle(ATTRIBUTE_CANDIDATES_PATH)

with open(CATEGORY_META_PATH, "wb") as file:
    pickle.dump(
        category_meta,
        file,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

with open(CATEGORY_ATTRIBUTES_PATH, "wb") as file:
    pickle.dump(
        category_attributes,
        file,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

print("Сохранено:")
print("-", ATTRIBUTE_STATS_PATH)
print("-", ATTRIBUTE_CANDIDATES_PATH)
print("-", CATEGORY_ATTRIBUTES_PATH)
print("-", CATEGORY_META_PATH)


### Постобработка схемы атрибутов

В исходной схеме встречались смешанные поля, например  
`Оперативная и встроенная память`. Такие поля обрабатывались вручную. Очищенный файл уже лежит в репе.

Для финального проекта используется заранее проверенный и очищенный файл
`category_attributes_by_subject_id.pkl`. Он должен быть помещён в `artifacts/`
до запуска инференса.


In [ ]:
if not CATEGORY_ATTRIBUTES_PATH.exists():
    raise FileNotFoundError(
        f"Не найдена очищенная схема: {CATEGORY_ATTRIBUTES_PATH}"
    )

with CATEGORY_ATTRIBUTES_PATH.open("rb") as file:
    cleaned_category_attributes = pickle.load(file)

print(
    "Очищенная схема найдена:",
    CATEGORY_ATTRIBUTES_PATH,
    "| категорий:",
    len(cleaned_category_attributes),
)


Очищенный pickle перезаписывает ранее построенный словарь по тому же пути. Следующие ячейки используют именно эту версию схемы.


## 7. Подготовка weak-supervised датасета для GLiNER2

Каталог содержит реальные значения характеристик, но исходный ноутбук сохранял только агрегаты уровня `категория × атрибут`. Ниже строится недостающая таблица уровня `SKU × атрибут × значение`, после чего значения ищутся в запросах, названиях и описаниях.

Разметка намеренно консервативна:

- используются только атрибуты из очищенной схемы категории;
- для дублей предпочитаются `vendor/model`, затем `*_search`, затем обычные `param`;
- числовое значение без единицы измерения не размечается;
- неоднозначные и пересекающиеся spans удаляются;
- train/validation/test разделяются по `sku_id`;
- одинаковый текст не может попасть в разные части датасета.


In [ ]:
MAX_QUERY_EXAMPLES_PER_SKU = 5
MAX_VALUE_CHARS = 160
MAX_DESCRIPTION_WINDOW_CHARS = 700
DESCRIPTION_WINDOW_MARGIN = 180
MAX_TRAIN_EXAMPLES_PER_CATEGORY_SOURCE = 2_000
MAX_EVAL_EXAMPLES_PER_CATEGORY_SOURCE = 300

VAL_SHARE = 0.10
TEST_SHARE = 0.10
MIN_SKUS_FOR_HOLDOUT_SPLIT = 10

CORE_ATTRIBUTES = {"Бренд", "Модель"}
LOW_INFORMATION_VALUES = {
    "да",
    "нет",
    "есть",
    "отсутствует",
    "не указано",
    "не применимо",
    "-",
    "—",
    "n/a",
    "none",
    "nan",
}

# Только однозначные переименования. Смешанные поля здесь не разделяются.
ATTRIBUTE_NAME_ALIASES = {
    "Оперативная память": "Оперативная память (RAM)",
    "Объем оперативной памяти": "Оперативная память (RAM)",
    "Объём оперативной памяти": "Оперативная память (RAM)",
    "Встроенная память": "Встроенная память (ROM)",
    "Объем встроенной памяти": "Встроенная память (ROM)",
    "Объём встроенной памяти": "Встроенная память (ROM)",
}

UNIT_ALIASES = {
    "гб": ["гб", "gb"],
    "тб": ["тб", "tb"],
    "мб": ["мб", "mb"],
    "кб": ["кб", "kb"],
    "мач": ["мач", "mah"],
    "вт": ["вт", "w"],
    "квт": ["квт", "kw"],
    "гц": ["гц", "hz"],
    "кг": ["кг", "kg"],
    "г": ["г", "g"],
    "см": ["см", "cm"],
    "мм": ["мм", "mm"],
    "м": ["м", "m"],
    "л": ["л", "l"],
    "мл": ["мл", "ml"],
    "дюйм": ["дюйм", "дюйма", "дюймов", '"', "″"],
    "об/мин": ["об/мин", "rpm"],
    "мбит/с": ["мбит/с", "mbps"],
}

print("Путь SKU-level значений:", SKU_ATTRIBUTE_VALUES_PATH)
print("Путь weak labels:", GLINER_WEAK_LABELS_PATH)


### 7.1. Таблица `SKU × атрибут × значение`

Для каждого SKU остаются только значения максимального приоритета конкретного атрибута. Единица измерения сохраняется отдельно и добавляется к поверхности значения, когда её нет в `#text`.


In [ ]:
from collections import defaultdict


def normalize_catalog_value(value):
    """Нормализация значения для дедупликации и сопоставления."""
    if value is None:
        return ""

    text = unicodedata.normalize("NFKC", str(value))
    text = html.unescape(text)
    text = text.replace("\xa0", " ")
    text = text.lower().replace("ё", "е")
    text = re.sub(r"[\u200b-\u200d\ufeff]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_unit(value):
    unit = normalize_catalog_value(value)
    unit = unit.strip(". ")
    return unit or None


def canonical_attribute_name(raw_name, allowed_attributes):
    """Приводит имя к очищенной production-схеме категории."""
    attribute_name = normalize_attribute_name(raw_name)

    if not attribute_name:
        return None

    if attribute_name in allowed_attributes:
        return attribute_name

    alias = ATTRIBUTE_NAME_ALIASES.get(attribute_name)

    if alias in allowed_attributes:
        return alias

    return None


def append_unit_if_missing(value, unit):
    value = str(value).strip()
    unit = normalize_unit(unit)

    if not value or not unit:
        return value

    value_normalized = normalize_catalog_value(value)
    unit_variants = UNIT_ALIASES.get(unit, [unit])

    if any(
        re.search(
            rf"(?<!\w){re.escape(alias)}(?!\w)",
            value_normalized,
            flags=re.IGNORECASE,
        )
        for alias in unit_variants
    ):
        return value

    return f"{value} {unit}".strip()


def catalog_value_is_usable(attribute_name, value, unit):
    normalized = normalize_catalog_value(value)

    if not normalized or normalized in LOW_INFORMATION_VALUES:
        return False

    if len(normalized) > MAX_VALUE_CHARS:
        return False

    if not re.search(r"[\w\d]", normalized, flags=re.UNICODE):
        return False

    # Чистое число без единицы слишком неоднозначно для weak supervision.
    numeric_only = bool(
        re.fullmatch(
            r"[+\-]?\d+(?:[.,]\d+)?(?:\s*[-–—]\s*\d+(?:[.,]\d+)?)?",
            normalized,
        )
    )

    if numeric_only and not normalize_unit(unit):
        return False

    if attribute_name in CORE_ATTRIBUTES:
        # Разрешаем короткие бренды вроде LG и модели вроде X5.
        return len(normalized) >= 2 and not normalized.isdigit()

    return len(normalized) >= 3


with open(CATEGORY_ATTRIBUTES_PATH, "rb") as file:
    category_attributes = pickle.load(file)

category_attributes = {
    normalize_id(category_id): list(dict.fromkeys(attributes))
    for category_id, attributes in category_attributes.items()
}

allowed_attributes_by_category = {
    category_id: set(attributes)
    for category_id, attributes in category_attributes.items()
}

sku_attribute_rows = []

for offer in tqdm(offers, desc="SKU-level значения атрибутов"):
    if not isinstance(offer, dict):
        continue

    sku_id = normalize_id(offer.get("@id"))
    category_id = sku_to_subject.get(sku_id)

    if category_id is None:
        continue

    allowed_attributes = allowed_attributes_by_category.get(category_id)

    if not allowed_attributes:
        continue

    # attribute -> records. После прохода оставим только max(priority).
    values_by_attribute = defaultdict(list)

    def add_value(raw_name, raw_value, *, unit=None, priority, source):
        attribute_name = canonical_attribute_name(
            raw_name,
            allowed_attributes,
        )

        if attribute_name is None:
            return

        for extracted_value in extract_values(raw_value):
            value_surface = append_unit_if_missing(
                extracted_value,
                unit,
            )

            if not catalog_value_is_usable(
                attribute_name,
                value_surface,
                unit,
            ):
                continue

            values_by_attribute[attribute_name].append(
                {
                    "value": value_surface,
                    "value_normalized": normalize_catalog_value(
                        value_surface
                    ),
                    "unit": normalize_unit(unit),
                    "priority": int(priority),
                    "source": source,
                }
            )

    add_value(
        "Бренд",
        offer.get("vendor"),
        priority=3,
        source="vendor",
    )
    add_value(
        "Модель",
        offer.get("model"),
        priority=3,
        source="model",
    )

    for parameter in ensure_list(offer.get("param")):
        if not isinstance(parameter, dict):
            continue

        raw_name = parameter.get("@name")

        if raw_name is None:
            continue

        is_search = bool(
            re.search(
                r"_search$",
                str(raw_name),
                flags=re.IGNORECASE,
            )
        )

        add_value(
            raw_name,
            parameter.get("#text"),
            unit=parameter.get("@unit"),
            priority=2 if is_search else 1,
            source="param_search" if is_search else "param",
        )

    for attribute_name, records in values_by_attribute.items():
        max_priority = max(record["priority"] for record in records)
        deduplicated = {}

        for record in records:
            if record["priority"] != max_priority:
                continue

            key = record["value_normalized"]
            deduplicated.setdefault(key, record)

        for record in deduplicated.values():
            sku_attribute_rows.append(
                {
                    "sku_id": sku_id,
                    "category_id": category_id,
                    "attribute_name": attribute_name,
                    **record,
                }
            )

sku_attribute_values = pd.DataFrame(sku_attribute_rows)

if sku_attribute_values.empty:
    raise RuntimeError(
        "SKU-level значения не построены. Проверь CATEGORY_ATTRIBUTES_PATH "
        "и пересечение SKU каталога с query_clicks."
    )

sku_attribute_values = (
    sku_attribute_values
    .drop_duplicates(
        [
            "sku_id",
            "category_id",
            "attribute_name",
            "value_normalized",
        ]
    )
    .reset_index(drop=True)
)

sku_attribute_values["priority"] = (
    sku_attribute_values["priority"].astype("int8")
)

sku_attribute_values.to_parquet(
    SKU_ATTRIBUTE_VALUES_PATH,
    index=False,
)

print("Сохранено:", SKU_ATTRIBUTE_VALUES_PATH)
print(f"Строк: {len(sku_attribute_values):,}")
print(f"SKU: {sku_attribute_values['sku_id'].nunique():,}")
print(f"Категорий: {sku_attribute_values['category_id'].nunique():,}")
print(f"Атрибутов: {sku_attribute_values['attribute_name'].nunique():,}")

display(sku_attribute_values.head(10))


### 7.2. Поиск high-precision spans

Поиск выполняется по нормализованным текстам, поэтому координаты относятся ровно к строке, которая попадёт в JSONL. Для значений с единицами создаются только безопасные варианты написания: с пробелом/без пробела и русские/английские обозначения единиц.


In [ ]:
def normalized_unit_aliases(unit):
    unit = normalize_unit(unit)
    return UNIT_ALIASES.get(unit, [unit] if unit else [])


def build_surface_variants(value, unit=None):
    """Строит консервативные варианты поверхности каталожного значения."""
    base = normalize_catalog_value(value)

    if not base:
        return []

    variants = {base}
    unit_aliases = normalized_unit_aliases(unit)

    if unit_aliases and re.search(r"\d", base):
        numeric_part = base

        # Снимаем единицу с конца, чтобы затем подставить допустимые aliases.
        for alias in sorted(unit_aliases, key=len, reverse=True):
            patterns = [
                rf"\s+{re.escape(alias)}$",
                rf"(?<=\d){re.escape(alias)}$",
            ]

            stripped = numeric_part

            for pattern in patterns:
                stripped = re.sub(pattern, "", stripped).strip()

            if stripped != numeric_part:
                numeric_part = stripped
                break

        if numeric_part and re.search(r"\d", numeric_part):
            numeric_forms = {
                numeric_part,
                numeric_part.replace(",", "."),
                numeric_part.replace(".", ","),
            }

            for numeric_form in numeric_forms:
                for alias in unit_aliases:
                    variants.add(f"{numeric_form} {alias}")
                    variants.add(f"{numeric_form}{alias}")

    # Частый формат 256 ГБ -> 256ГБ даже если @unit отсутствовал.
    compact = re.sub(
        r"(?<=\d)\s+(?=[a-zа-я\"″])",
        "",
        base,
        flags=re.IGNORECASE,
    )
    variants.add(compact)

    return sorted(
        {variant for variant in variants if variant},
        key=lambda item: (-len(item), item),
    )


def boundary_is_valid(text, start, end, surface):
    left_requires_boundary = bool(surface and surface[0].isalnum())
    right_requires_boundary = bool(surface and surface[-1].isalnum())

    if (
        left_requires_boundary
        and start > 0
        and (text[start - 1].isalnum() or text[start - 1] == "_")
    ):
        return False

    if (
        right_requires_boundary
        and end < len(text)
        and (text[end].isalnum() or text[end] == "_")
    ):
        return False

    return True


def find_surface_occurrences(text, surface, max_occurrences=3):
    occurrences = []
    cursor = 0

    while len(occurrences) < max_occurrences:
        start = text.find(surface, cursor)

        if start < 0:
            break

        end = start + len(surface)

        if boundary_is_valid(text, start, end, surface):
            occurrences.append((start, end))

        cursor = start + 1

    return occurrences


def resolve_mentions(mentions):
    """Удаляет неоднозначные одинаковые spans и пересечения."""
    deduplicated = {}

    for mention in mentions:
        key = (
            mention["attribute"],
            int(mention["start"]),
            int(mention["end"]),
        )
        current = deduplicated.get(key)

        if current is None or mention["priority"] > current["priority"]:
            deduplicated[key] = mention

    mentions = list(deduplicated.values())

    labels_by_span = defaultdict(set)

    for mention in mentions:
        labels_by_span[
            (mention["start"], mention["end"])
        ].add(mention["attribute"])

    ambiguous_spans = {
        span
        for span, labels in labels_by_span.items()
        if len(labels) > 1
    }

    mentions = [
        mention
        for mention in mentions
        if (mention["start"], mention["end"]) not in ambiguous_spans
    ]

    # GLiNER2 NER-разметке безопаснее дать непересекающиеся сущности.
    ranked = sorted(
        mentions,
        key=lambda mention: (
            -int(mention["priority"]),
            -(int(mention["end"]) - int(mention["start"])),
            int(mention["start"]),
            mention["attribute"],
        ),
    )

    selected = []

    for candidate in ranked:
        overlaps = any(
            not (
                candidate["end"] <= existing["start"]
                or candidate["start"] >= existing["end"]
            )
            for existing in selected
        )

        if not overlaps:
            selected.append(candidate)

    return sorted(selected, key=lambda mention: (mention["start"], mention["end"]))


def match_sku_values(text, value_records):
    text = clean_text(text, remove_html=False)

    if not text:
        return text, []

    mentions = []

    for record in value_records:
        for surface in build_surface_variants(
            record["value"],
            record.get("unit"),
        ):
            for start, end in find_surface_occurrences(text, surface):
                mentions.append(
                    {
                        "attribute": record["attribute_name"],
                        "text": text[start:end],
                        "start": int(start),
                        "end": int(end),
                        "priority": int(record["priority"]),
                        "catalog_value": record["value"],
                        "match_surface": surface,
                    }
                )

    return text, resolve_mentions(mentions)


def description_windows(text, mentions):
    """Вырезает компактные окна вокруг размеченных сущностей."""
    if not mentions:
        return []

    if len(text) <= MAX_DESCRIPTION_WINDOW_CHARS:
        return [(text, mentions, 0)]

    intervals = []

    for mention in mentions:
        start = max(0, mention["start"] - DESCRIPTION_WINDOW_MARGIN)
        end = min(len(text), mention["end"] + DESCRIPTION_WINDOW_MARGIN)

        # Ограничиваем размер, сохраняя сущность внутри окна.
        if end - start > MAX_DESCRIPTION_WINDOW_CHARS:
            center = (mention["start"] + mention["end"]) // 2
            start = max(0, center - MAX_DESCRIPTION_WINDOW_CHARS // 2)
            end = min(len(text), start + MAX_DESCRIPTION_WINDOW_CHARS)
            start = max(0, end - MAX_DESCRIPTION_WINDOW_CHARS)

        intervals.append([start, end])

    intervals.sort()
    merged = []

    for start, end in intervals:
        if merged and start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])

    windows = []

    for start, end in merged:
        # Сдвигаем границы к пробелам, не выходя далеко за лимит.
        while start > 0 and text[start - 1] != " " and end - start < MAX_DESCRIPTION_WINDOW_CHARS:
            start -= 1
        while end < len(text) and text[end] != " " and end - start < MAX_DESCRIPTION_WINDOW_CHARS:
            end += 1

        raw_window = text[start:end]
        left_trim = len(raw_window) - len(raw_window.lstrip())
        window = raw_window.strip()
        effective_start = start + left_trim
        effective_end = effective_start + len(window)

        window_mentions = []

        for mention in mentions:
            if mention["start"] >= effective_start and mention["end"] <= effective_end:
                adjusted = dict(mention)
                adjusted["start"] -= effective_start
                adjusted["end"] -= effective_start
                adjusted["text"] = window[
                    adjusted["start"]:adjusted["end"]
                ]
                window_mentions.append(adjusted)

        if window and window_mentions:
            windows.append((window, window_mentions, effective_start))

    return windows


def mentions_to_entities(mentions):
    entities = defaultdict(list)

    for mention in mentions:
        value = mention["text"]

        if value not in entities[mention["attribute"]]:
            entities[mention["attribute"]].append(value)

    return dict(sorted(entities.items()))


### 7.3. Формирование примеров и безопасный split

Основная целевая поверхность — поисковые запросы. Для разнообразия добавляются названия товаров и короткие окна описаний. Один SKU целиком относится только к одной части датасета. Тексты с конфликтующей разметкой полностью исключаются.


In [ ]:
import hashlib


sku_value_records = defaultdict(list)

for row in sku_attribute_values.itertuples(index=False):
    sku_value_records[str(row.sku_id)].append(
        {
            "sku_id": str(row.sku_id),
            "category_id": str(row.category_id),
            "attribute_name": row.attribute_name,
            "value": row.value,
            "value_normalized": row.value_normalized,
            "unit": row.unit,
            "priority": int(row.priority),
            "source": row.source,
        }
    )

eligible_skus = set(sku_value_records)
eligible_sku_numeric = {
    int(sku_id)
    for sku_id in eligible_skus
    if str(sku_id).isdigit()
}


def stable_uint64(value):
    digest = hashlib.sha1(str(value).encode("utf-8")).digest()
    return int.from_bytes(digest[:8], byteorder="big", signed=False)


def build_sku_split_table(values_df):
    sku_table = (
        values_df[["sku_id", "category_id"]]
        .drop_duplicates("sku_id")
        .copy()
    )

    sku_table["split_order"] = [
        stable_uint64(f"{category_id}:{sku_id}:{RANDOM_STATE}")
        for sku_id, category_id in zip(
            sku_table["sku_id"],
            sku_table["category_id"],
        )
    ]

    split_rows = []

    for category_id, group in sku_table.groupby("category_id", sort=False):
        group = group.sort_values("split_order").copy()
        n_skus = len(group)
        group["split"] = "train"

        if n_skus >= MIN_SKUS_FOR_HOLDOUT_SPLIT:
            n_test = max(1, int(round(n_skus * TEST_SHARE)))
            n_val = max(1, int(round(n_skus * VAL_SHARE)))

            # Всегда оставляем хотя бы один SKU в train.
            overflow = max(0, n_test + n_val - (n_skus - 1))

            if overflow:
                n_val = max(0, n_val - overflow)

            group.iloc[:n_test, group.columns.get_loc("split")] = "test"
            group.iloc[n_test:n_test + n_val, group.columns.get_loc("split")] = "val"

        split_rows.append(group)

    return pd.concat(split_rows, ignore_index=True)


sku_split_table = build_sku_split_table(sku_attribute_values)
sku_to_split = dict(
    zip(
        sku_split_table["sku_id"].astype(str),
        sku_split_table["split"],
    )
)

print("SKU по split:")
display(sku_split_table["split"].value_counts().to_frame("sku"))

# Необязательная защита от ранее созданного real-query holdout.
holdout_sku_paths = [
    ARTIFACTS_DIR / "proxy_gold_holdout_skus.csv.gz",
    DATA_DIR / "proxy_gold_holdout_skus.csv.gz",
]
holdout_skus = set()

for path in holdout_sku_paths:
    if not path.exists():
        continue

    holdout_frame = pd.read_csv(path)
    sku_column = next(
        (
            column
            for column in ["sku_id", "representative_sku_id"]
            if column in holdout_frame.columns
        ),
        None,
    )

    if sku_column is not None:
        holdout_skus.update(
            holdout_frame[sku_column]
            .dropna()
            .map(normalize_id)
            .dropna()
        )

if holdout_skus:
    print(f"Исключаем SKU из external holdout: {len(holdout_skus):,}")


weak_rows = []
source_counters = Counter()


def add_weak_example(*, sku_id, category_id, source, text):
    sku_id = normalize_id(sku_id)
    category_id = normalize_id(category_id)

    if sku_id is None or sku_id in holdout_skus:
        return

    value_records = sku_value_records.get(sku_id)

    if not value_records:
        return

    normalized_text, mentions = match_sku_values(text, value_records)

    if not mentions:
        return

    windows = (
        description_windows(normalized_text, mentions)
        if source == "description"
        else [(normalized_text, mentions, 0)]
    )

    for window_index, (window_text, window_mentions, source_offset) in enumerate(windows):
        entities = mentions_to_entities(window_mentions)

        if not entities:
            continue

        spans_payload = [
            {
                "attribute": mention["attribute"],
                "text": mention["text"],
                "start": int(mention["start"]),
                "end": int(mention["end"]),
                "priority": int(mention["priority"]),
                "catalog_value": mention["catalog_value"],
            }
            for mention in window_mentions
        ]

        # Последняя проверка инварианта span -> substring.
        assert all(
            window_text[item["start"]:item["end"]] == item["text"]
            for item in spans_payload
        )

        entities_json = json.dumps(
            entities,
            ensure_ascii=False,
            sort_keys=True,
        )

        weak_rows.append(
            {
                "example_id": (
                    f"{source}:{sku_id}:{source_counters[source]}:{window_index}"
                ),
                "sku_id": sku_id,
                "category_id": category_id,
                "source": source,
                "text": window_text,
                "entities_json": entities_json,
                "spans_json": json.dumps(
                    spans_payload,
                    ensure_ascii=False,
                ),
                "n_entities": int(len(spans_payload)),
                "n_attributes": int(len(entities)),
                "source_offset": int(source_offset),
                "split": sku_to_split.get(sku_id, "train"),
            }
        )
        source_counters[source] += 1


# Запросы: ограничиваем число уникальных текстов на SKU.
query_source = (
    query_rows.loc[
        query_rows["sku_id"].isin(eligible_sku_numeric),
        ["sku_id", "target", "text"],
    ]
    .dropna(subset=["sku_id", "target", "text"])
    .drop_duplicates(["sku_id", "text"])
    .groupby("sku_id", sort=False, as_index=False, group_keys=False)
    .head(MAX_QUERY_EXAMPLES_PER_SKU)
    .reset_index(drop=True)
)

for row in tqdm(
    query_source.itertuples(index=False),
    total=len(query_source),
    desc="Weak labels: query",
):
    add_weak_example(
        sku_id=row.sku_id,
        category_id=row.target,
        source="query",
        text=row.text,
    )

# Названия карточек.
title_source = (
    cards.loc[
        cards["sku_id"].isin(eligible_sku_numeric),
        ["sku_id", "target", "title_clean"],
    ]
    .dropna(subset=["sku_id", "target", "title_clean"])
    .drop_duplicates(["sku_id", "title_clean"])
)

for row in tqdm(
    title_source.itertuples(index=False),
    total=len(title_source),
    desc="Weak labels: title",
):
    add_weak_example(
        sku_id=row.sku_id,
        category_id=row.target,
        source="title",
        text=row.title_clean,
    )

# Описания: в датасет попадут только окна вокруг найденных значений.
description_source = (
    cards.loc[
        cards["sku_id"].isin(eligible_sku_numeric),
        ["sku_id", "target", "description_clean"],
    ]
    .dropna(subset=["sku_id", "target", "description_clean"])
    .drop_duplicates(["sku_id", "description_clean"])
)

for row in tqdm(
    description_source.itertuples(index=False),
    total=len(description_source),
    desc="Weak labels: description",
):
    add_weak_example(
        sku_id=row.sku_id,
        category_id=row.target,
        source="description",
        text=row.description_clean,
    )

weak_labels = pd.DataFrame(weak_rows)

if weak_labels.empty:
    raise RuntimeError("Weak-supervised примеры не построены.")

weak_labels["text_key"] = weak_labels["text"].map(
    normalize_catalog_value
)
weak_labels["entity_signature"] = weak_labels["entities_json"]

# Одинаковый текст с разной разметкой является неоднозначным — удаляем целиком.
conflicting_text_keys = set(
    weak_labels.groupby("text_key")["entity_signature"]
    .nunique()
    .loc[lambda counts: counts > 1]
    .index
)

weak_labels = weak_labels.loc[
    ~weak_labels["text_key"].isin(conflicting_text_keys)
].copy()

# Один текст оставляем ровно в одном split. Приоритет: test -> val -> train.
split_priority = {"test": 0, "val": 1, "train": 2}
source_priority = {"query": 0, "title": 1, "description": 2}

weak_labels["split_priority"] = weak_labels["split"].map(split_priority)
weak_labels["source_priority"] = weak_labels["source"].map(source_priority)
weak_labels["stable_order"] = weak_labels["text_key"].map(stable_uint64)

weak_labels = (
    weak_labels
    .sort_values(
        [
            "text_key",
            "split_priority",
            "source_priority",
            "n_entities",
            "stable_order",
        ],
        ascending=[True, True, True, False, True],
    )
    .drop_duplicates("text_key")
    .reset_index(drop=True)
)

# Ограничиваем доминирование популярных категорий.
def cap_group(frame, cap):
    return (
        frame.sort_values("stable_order")
        .groupby(["category_id", "source"], sort=False, group_keys=False)
        .head(cap)
    )

train_part = cap_group(
    weak_labels[weak_labels["split"] == "train"],
    MAX_TRAIN_EXAMPLES_PER_CATEGORY_SOURCE,
)
val_part = cap_group(
    weak_labels[weak_labels["split"] == "val"],
    MAX_EVAL_EXAMPLES_PER_CATEGORY_SOURCE,
)
test_part = cap_group(
    weak_labels[weak_labels["split"] == "test"],
    MAX_EVAL_EXAMPLES_PER_CATEGORY_SOURCE,
)

weak_labels = pd.concat(
    [train_part, val_part, test_part],
    ignore_index=True,
)

weak_labels = weak_labels.drop(
    columns=[
        "split_priority",
        "source_priority",
        "stable_order",
    ]
)

# Финальные проверки leakage и структуры.
assert weak_labels["text_key"].is_unique
assert (
    weak_labels.groupby("sku_id")["split"].nunique().max() == 1
)
assert set(weak_labels["split"]).issubset({"train", "val", "test"})
assert weak_labels["n_entities"].min() >= 1

weak_labels.to_parquet(
    GLINER_WEAK_LABELS_PATH,
    index=False,
)

print("Сохранено:", GLINER_WEAK_LABELS_PATH)
print(f"Примеров: {len(weak_labels):,}")
print(f"Удалено конфликтующих текстов: {len(conflicting_text_keys):,}")
print()
print("По split:")
display(weak_labels["split"].value_counts().to_frame("examples"))
print("По источнику:")
display(weak_labels["source"].value_counts().to_frame("examples"))


### 7.4. Экспорт JSONL и строгая валидация

GLiNER2 принимает JSONL в формате `input/output`. Отдельный parquet сохраняет SKU, категорию, источник и span-координаты для аудита.


In [ ]:
def write_gliner_jsonl(frame, path):
    path = Path(path)

    with path.open("w", encoding="utf-8") as file:
        for row in frame.itertuples(index=False):
            entities = json.loads(row.entities_json)

            payload = {
                "input": row.text,
                "output": {
                    "entities": entities,
                },
            }

            file.write(
                json.dumps(payload, ensure_ascii=False)
                + "\n"
            )


split_to_path = {
    "train": GLINER_TRAIN_PATH,
    "val": GLINER_VAL_PATH,
    "test": GLINER_TEST_PATH,
}

for split_name, output_path in split_to_path.items():
    subset = weak_labels[
        weak_labels["split"] == split_name
    ].copy()

    if subset.empty:
        raise RuntimeError(
            f"Split {split_name!r} пуст. Уменьши "
            "MIN_SKUS_FOR_HOLDOUT_SPLIT или проверь coverage."
        )

    write_gliner_jsonl(subset, output_path)
    print(f"{split_name}: {len(subset):,} -> {output_path}")

# Проверяем, что каждая размеченная поверхность реально содержится в input.
invalid_surface_rows = []

for row in weak_labels.itertuples(index=False):
    entities = json.loads(row.entities_json)

    for attribute, values in entities.items():
        for value in values:
            if value not in row.text:
                invalid_surface_rows.append(
                    {
                        "example_id": row.example_id,
                        "attribute": attribute,
                        "value": value,
                    }
                )

if invalid_surface_rows:
    raise AssertionError(
        f"Найдено {len(invalid_surface_rows)} сущностей, "
        "которых нет в тексте."
    )

from gliner2.training.data import TrainingDataset
import inspect

validation_summaries = {}

for split_name, output_path in split_to_path.items():
    dataset = TrainingDataset.load(str(output_path))

    validate_signature = inspect.signature(dataset.validate)
    supported_parameters = validate_signature.parameters

    validate_kwargs = {}

    if "strict" in supported_parameters:
        validate_kwargs["strict"] = True

    if "raise_on_error" in supported_parameters:
        validate_kwargs["raise_on_error"] = True

    validation_result = dataset.validate(**validate_kwargs)
    validation_summaries[split_name] = {
        "examples": len(dataset),
        "result": str(validation_result),
    }

    print(
        f"{split_name}: "
        f"{len(dataset)} примеров, "
        f"validate={validation_result}"
    )

print("Строгая валидация GLiNER2 пройдена для всех split.")

attribute_counts = Counter()

for entities_json in weak_labels["entities_json"]:
    entities = json.loads(entities_json)

    for attribute, values in entities.items():
        attribute_counts[attribute] += len(values)

report_payload = {
    "sku_attribute_rows": int(len(sku_attribute_values)),
    "sku_with_values": int(sku_attribute_values["sku_id"].nunique()),
    "weak_examples": int(len(weak_labels)),
    "conflicting_texts_removed": int(len(conflicting_text_keys)),
    "split_counts": {
        str(key): int(value)
        for key, value in weak_labels["split"].value_counts().items()
    },
    "source_counts": {
        str(key): int(value)
        for key, value in weak_labels["source"].value_counts().items()
    },
    "category_count": int(weak_labels["category_id"].nunique()),
    "attribute_count": int(len(attribute_counts)),
    "top_attributes": attribute_counts.most_common(50),
    "validation": validation_summaries,
    "config": {
        "max_query_examples_per_sku": MAX_QUERY_EXAMPLES_PER_SKU,
        "max_train_examples_per_category_source": (
            MAX_TRAIN_EXAMPLES_PER_CATEGORY_SOURCE
        ),
        "max_eval_examples_per_category_source": (
            MAX_EVAL_EXAMPLES_PER_CATEGORY_SOURCE
        ),
        "random_state": RANDOM_STATE,
    },
}

with GLINER_DATASET_REPORT_PATH.open("w", encoding="utf-8") as file:
    json.dump(
        report_payload,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Отчёт:", GLINER_DATASET_REPORT_PATH)
display(
    pd.DataFrame(
        attribute_counts.most_common(30),
        columns=["attribute", "mentions"],
    )
)


## 8. Эксперимент: LoRA-дообучение GLiNER2

Эксперимент выполнялся на 221 438 weak-supervised примерах. В финальной оценке адаптер ухудшил Span F1 с **0,7035** до **0,6596**, поэтому production-like вариант проекта использует базовый GLiNER2. Вероятно, это произошло из-за качества данных. Но увы, кроме как програмным методом, данные разметить не могу.

Раздел сохранён для воспроизводимости отрицательного результата. Обучение отключено по умолчанию.


In [ ]:
from gliner2 import GLiNER2
from gliner2.training.trainer import GLiNER2Trainer, TrainingConfig


RUN_GLINER2_TRAINING = False

if RUN_GLINER2_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError("Для LoRA-эксперимента требуется CUDA GPU.")

    config_kwargs = {
        "output_dir": str(GLINER_ADAPTER_DIR),
        "experiment_name": "marketplace_attributes",
        "num_epochs": 3,
        "batch_size": 8,
        "eval_batch_size": 8,
        "gradient_accumulation_steps": 2,
        "encoder_lr": 1e-5,
        "task_lr": 5e-4,
        "weight_decay": 0.01,
        "warmup_ratio": 0.10,
        "scheduler_type": "cosine",
        "max_grad_norm": 1.0,
        "fp16": True,
        "bf16": False,
        "eval_strategy": "epoch",
        "save_total_limit": 2,
        "save_best": True,
        "metric_for_best": "eval_loss",
        "greater_is_better": False,
        "early_stopping": True,
        "early_stopping_patience": 2,
        "logging_steps": 50,
        "report_to_wandb": False,
        "num_workers": 2,
        "use_lora": True,
        "lora_r": 16,
        "lora_alpha": 32.0,
        "lora_dropout": 0.05,
        "lora_target_modules": ["encoder"],
        "save_adapter_only": True,
    }

    # API TrainingConfig отличается между версиями GLiNER2.
    supported = set(inspect.signature(TrainingConfig).parameters)
    filtered_kwargs = {
        key: value
        for key, value in config_kwargs.items()
        if key in supported
    }
    skipped = sorted(set(config_kwargs) - set(filtered_kwargs))
    if skipped:
        print("Неподдерживаемые параметры пропущены:", skipped)

    gliner_train_model = GLiNER2.from_pretrained(
        GLINER_BASE_MODEL,
        map_location="cuda",
    )
    gliner_training_config = TrainingConfig(**filtered_kwargs)
    gliner_trainer = GLiNER2Trainer(
        model=gliner_train_model,
        config=gliner_training_config,
    )

    training_results = gliner_trainer.train(
        train_data=str(GLINER_TRAIN_PATH),
        eval_data=str(GLINER_VAL_PATH),
    )

    serializable_results = {
        key: (
            int(value) if isinstance(value, np.integer)
            else float(value) if isinstance(value, np.floating)
            else value if isinstance(value, (str, int, float, bool, type(None), list, dict))
            else str(value)
        )
        for key, value in training_results.items()
    }
    serializable_results.update(
        {
            "base_model": GLINER_BASE_MODEL,
            "adapter_path": str(GLINER_ADAPTER_FINAL_DIR),
            "cuda_device": torch.cuda.get_device_name(0),
        }
    )

    with GLINER_TRAINING_REPORT_PATH.open("w", encoding="utf-8") as file:
        json.dump(serializable_results, file, ensure_ascii=False, indent=2)

    print("Адаптер:", GLINER_ADAPTER_FINAL_DIR)
else:
    print("LoRA-обучение пропущено.")


### 8.1. Smoke test адаптера

Проверяется, что адаптер загружается поверх исходной multilingual-модели и возвращает spans в ожидаемом формате.


In [ ]:
if GLINER_ADAPTER_FINAL_DIR.exists():
    tuned_extractor = GLiNER2.from_pretrained(
        GLINER_BASE_MODEL,
        map_location=DEVICE,
    )
    tuned_extractor.load_adapter(str(GLINER_ADAPTER_FINAL_DIR))

    smoke_result = tuned_extractor.extract_entities(
        "Samsung Galaxy S26 256 ГБ",
        ["Бренд", "Модель", "Встроенная память (ROM)"],
        threshold=0.2,
        include_confidence=True,
        include_spans=True,
    )
    print(smoke_result)
else:
    print("LoRA-адаптер отсутствует — smoke test пропущен.")


## 9. GLiNER2 и единый инференс

Для инференса заново загружаются сохранённые артефакты. Это проверяет, что итоговая функция не зависит от временных объектов, оставшихся после обучения.

Если для предсказанной категории не удалось построить схему, используется минимальный fallback из атрибутов **Бренд** и **Модель**.


In [ ]:
from gliner2 import GLiNER2


model = fasttext.load_model(str(MODEL_PATH))

with open(CATEGORY_ATTRIBUTES_PATH, "rb") as file:
    category_attributes = pickle.load(file)

with open(CATEGORY_META_PATH, "rb") as file:
    category_meta = pickle.load(file)

category_attributes = {
    normalize_id(category_id): list(
        dict.fromkeys(attributes)
    )
    for category_id, attributes in category_attributes.items()
}

# Статистический prior нужен только для ранжирования shortlist.
# Если файл отсутствует, selector использует порядок атрибутов в production-схеме.
attribute_priority_lookup = {}

if Path(ATTRIBUTE_CANDIDATES_PATH).exists():
    attribute_priority_frame = pd.read_pickle(
        ATTRIBUTE_CANDIDATES_PATH
    ).copy()

    attribute_priority_frame["category_id"] = (
        attribute_priority_frame["category_id"]
        .map(normalize_id)
    )

    for row in attribute_priority_frame.itertuples(index=False):
        category_id = normalize_id(row.category_id)
        attribute_name = str(row.attribute_name)

        attribute_priority_lookup.setdefault(
            category_id,
            {},
        )[attribute_name] = {
            "rank": int(getattr(row, "attribute_rank", 10_000)),
            "score": float(getattr(row, "score", 0.0)),
            "coverage": float(getattr(row, "coverage", 0.0)),
        }

extractor = GLiNER2.from_pretrained(
    GLINER_BASE_MODEL,
    map_location=DEVICE,
)

GLINER_VARIANT = "base"
USE_GLINER_ADAPTER_FOR_INFERENCE = False  # финальное решение: base GLiNER2

if (
    USE_GLINER_ADAPTER_FOR_INFERENCE
    and GLINER_ADAPTER_FINAL_DIR.exists()
):
    extractor.load_adapter(
        str(GLINER_ADAPTER_FINAL_DIR)
    )
    GLINER_VARIANT = "marketplace_lora"

print("Модель fastText загружена.")
print("Схем категорий:", len(category_attributes))
print("Категорий с prior:", len(attribute_priority_lookup))
print("GLiNER2 variant:", GLINER_VARIANT)
print("GLiNER2 загружен.")


### Вспомогательные функции

Классификатор возвращает top-k категорий, но основной результат всегда строится по top-1. Top-k сохраняется в ответе для диагностики. Из результата GLiNER2 дополнительно формируется плоский список фактов.


In [ ]:
GENERIC_ATTRIBUTES = [
    "Бренд",
    "Модель",
]


def normalize_query_for_fasttext(query):
    """Использует ту же нормализацию, что и обучающий датасет."""
    return clean_text(query, remove_html=False)


def predict_category(query, k=3):
    """Возвращает top-k категорий fastText."""
    normalized_query = normalize_query_for_fasttext(query)

    labels, probabilities = model.predict(
        normalized_query,
        k=k,
    )

    predictions = []

    for rank, (label, probability) in enumerate(
        zip(labels, probabilities),
        start=1,
    ):
        predictions.append(
            {
                "rank": rank,
                "category_id": normalize_id(
                    label.removeprefix("__label__")
                ),
                "confidence": float(probability),
            }
        )

    return predictions


def flatten_facts(facts):
    """Преобразует словарь сущностей в список записей."""
    flattened = []

    if not isinstance(facts, dict):
        return flattened

    for attribute_name, entities in facts.items():
        if not isinstance(entities, list):
            entities = [entities]

        for entity in entities:
            if isinstance(entity, dict):
                flattened.append(
                    {
                        "attribute": attribute_name,
                        "value": entity.get("text"),
                        "confidence": entity.get(
                            "confidence"
                        ),
                        "start": entity.get("start"),
                        "end": entity.get("end"),
                    }
                )
            else:
                flattened.append(
                    {
                        "attribute": attribute_name,
                        "value": entity,
                        "confidence": None,
                        "start": None,
                        "end": None,
                    }
                )

    return flattened


### Динамический shortlist атрибутов

Полная схема категории содержит в среднем около 25 labels. Это увеличивает вычислительную стоимость GLiNER2 и создаёт конкуренцию между похожими атрибутами.

Shortlist объединяет:

- обязательные `Бренд` и `Модель`;
- статистический prior атрибутов категории;
- слова-подсказки в запросе;
- единицы измерения (`ГБ`, `мАч`, `Гц`, `Вт`, `кг`, `см`, `МП`).

По итогам эксперимента используется **5 атрибутов**.


In [ ]:
SHORTLIST_STOPWORDS = {
    "и", "или", "для", "на", "в", "с", "по", "из", "от",
    "тип", "наличие", "поддержка", "режим", "функция",
    "характеристика", "значение", "максимальный", "минимальный",
}

# query_regex -> attribute_regex -> boost
ATTRIBUTE_QUERY_HINT_RULES = [
    (r"\b(оператив\w*|озу|ram)\b", r"оператив|\bram\b", 45.0, "RAM hint"),
    (r"\b(встроенн\w*\s+памят\w*|rom|накопител\w*|ssd|hdd)\b", r"встроенн.*памят|\brom\b|накопител|диск", 45.0, "storage hint"),
    (r"\b(аккумулятор\w*|батаре\w*|мач|mah)\b", r"аккумулятор|батаре|емкость", 40.0, "battery hint"),
    (r"\b(диагонал\w*|дюйм\w*|inch|inches)\b", r"диагонал|размер.*экран", 40.0, "diagonal hint"),
    (r"\b(разрешени\w*|4k|8k|full\s*hd|uhd|qhd|\d{3,4}\s*[xх×]\s*\d{3,4})\b", r"разрешени", 40.0, "resolution hint"),
    (r"\b(цвет\w*|черн\w*|бел\w*|красн\w*|син\w*|сер\w*|зелен\w*|золот\w*)\b", r"цвет", 32.0, "color hint"),
    (r"\b(вес\w*|масса)\b", r"вес|масса", 38.0, "weight hint"),
    (r"\b(высот\w*)\b", r"высот", 38.0, "height hint"),
    (r"\b(ширин\w*)\b", r"ширин", 38.0, "width hint"),
    (r"\b(глубин\w*)\b", r"глубин", 38.0, "depth hint"),
    (r"\b(длин\w*)\b", r"длин", 38.0, "length hint"),
    (r"\b(диаметр\w*)\b", r"диаметр", 38.0, "diameter hint"),
    (r"\b(мощност\w*|ватт\w*|киловатт\w*)\b", r"мощност|потребляем", 38.0, "power hint"),
    (r"\b(частот\w*|герц\w*|гц|hz)\b", r"частот|обновлен", 38.0, "frequency hint"),
    (r"\b(камера\w*|мегапиксел\w*|мп)\b", r"камер|мегапиксел|разрешение.*фото", 36.0, "camera hint"),
    (r"\b(sim|сим)\b", r"sim|сим", 35.0, "SIM hint"),
    (r"\b(процессор\w*|cpu|чип\w*)\b", r"процессор|чипсет", 35.0, "CPU hint"),
    (r"\b(ядр\w*)\b", r"ядер|ядр", 35.0, "core-count hint"),
    (r"\b(объем\w*|обьем\w*|литр\w*)\b", r"объем|обьем|вместимост", 28.0, "volume hint"),
    (r"\b(гаранти\w*)\b", r"гаранти", 35.0, "warranty hint"),
    (r"\b(страна|производств\w*)\b", r"страна|производств", 30.0, "country hint"),
]

ATTRIBUTE_UNIT_HINT_RULES = [
    (r"(?<!\w)\d+(?:[.,]\d+)?\s*(?:тб|tb|гб|gb|мб|mb)(?!\w)", r"памят|\bram\b|\brom\b|накопител|видеопамят|диск", 28.0, "memory unit"),
    (r"(?<!\w)\d+(?:[.,]\d+)?\s*(?:мач|mah)(?!\w)", r"аккумулятор|батаре|емкость", 42.0, "battery unit"),
    (r"(?<!\w)\d+(?:[.,]\d+)?\s*(?:гц|hz)(?!\w)", r"частот|обновлен", 34.0, "display frequency unit"),
    (r"(?<!\w)\d+(?:[.,]\d+)?\s*(?:кгц|khz|мгц|mhz|ггц|ghz)(?!\w)", r"частот|процессор|cpu", 32.0, "processor frequency unit"),
    (r"(?<!\w)\d+(?:[.,]\d+)?\s*[\"″]", r"диагонал|размер.*экран", 40.0, "inch quote"),
    (r"(?<!\w)\d+(?:[.,]\d+)?\s*(?:вт|w|квт|kw)(?!\w)", r"мощност|потребляем", 35.0, "power unit"),
    (r"(?<!\w)\d+(?:[.,]\d+)?\s*(?:кг|kg|г|гр|gram)(?!\w)", r"вес|масса|загрузк", 28.0, "mass unit"),
    (r"(?<!\w)\d+(?:[.,]\d+)?\s*(?:мм|mm|см|cm|м|meter)(?!\w)", r"высот|ширин|глубин|длин|диаметр|размер|толщин", 24.0, "length unit"),
    (r"(?<!\w)\d+(?:[.,]\d+)?\s*(?:л|литр|ml|мл)(?!\w)", r"объем|обьем|вместимост|резервуар", 28.0, "volume unit"),
    (r"(?<!\w)\d+(?:[.,]\d+)?\s*(?:об/мин|rpm)(?!\w)", r"оборот|скорост|отжим", 34.0, "rotation unit"),
    (r"(?<!\w)\d+(?:[.,]\d+)?\s*(?:мп|mp)(?!\w)", r"камер|мегапиксел|разрешение.*фото", 38.0, "camera unit"),
]


def normalize_shortlist_text(value):
    value = unicodedata.normalize("NFKC", str(value or ""))
    value = value.lower().replace("ё", "е")
    value = re.sub(r"[^\w\d]+", " ", value, flags=re.UNICODE)
    return re.sub(r"\s+", " ", value).strip()


def shortlist_tokens(value):
    return {
        token
        for token in normalize_shortlist_text(value).split()
        if len(token) >= 3 and token not in SHORTLIST_STOPWORDS
    }


def rank_attributes_for_query(query, category_id, attributes):
    """Возвращает диагностическую таблицу ranking для одной категории."""
    query_raw = str(query)
    query_normalized = normalize_shortlist_text(query_raw)
    query_token_set = shortlist_tokens(query_raw)
    category_id = normalize_id(category_id)
    priors = attribute_priority_lookup.get(category_id, {})

    rows = []

    for list_position, attribute_name in enumerate(attributes, start=1):
        attribute_normalized = normalize_shortlist_text(attribute_name)
        attribute_token_set = shortlist_tokens(attribute_name)
        prior = priors.get(attribute_name, {})
        rank = int(prior.get("rank", list_position))
        statistical_score = float(prior.get("score", 0.0))
        coverage = float(prior.get("coverage", 0.0))

        # Маленький prior: сигнал запроса должен иметь возможность изменить порядок.
        score = (
            3.0 / max(rank, 1)
            + 2.0 * statistical_score
            + 0.5 * coverage
        )
        reasons = [f"category_prior_rank={rank}"]

        if attribute_name in GENERIC_ATTRIBUTES:
            score += 100.0
            reasons.append("mandatory_generic")

        overlap = query_token_set & attribute_token_set
        if overlap:
            overlap_boost = 8.0 + 3.0 * len(overlap)
            score += overlap_boost
            reasons.append(
                "token_overlap=" + ",".join(sorted(overlap))
            )

        for query_pattern, attribute_pattern, boost, reason in ATTRIBUTE_QUERY_HINT_RULES:
            if (
                re.search(query_pattern, query_normalized, flags=re.IGNORECASE)
                and re.search(attribute_pattern, attribute_normalized, flags=re.IGNORECASE)
            ):
                score += boost
                reasons.append(reason)

        for query_pattern, attribute_pattern, boost, reason in ATTRIBUTE_UNIT_HINT_RULES:
            if (
                re.search(query_pattern, query_raw, flags=re.IGNORECASE)
                and re.search(attribute_pattern, attribute_normalized, flags=re.IGNORECASE)
            ):
                score += boost
                reasons.append(reason)

        rows.append(
            {
                "attribute": attribute_name,
                "score": float(score),
                "category_rank": rank,
                "list_position": list_position,
                "reasons": reasons,
            }
        )

    return sorted(
        rows,
        key=lambda row: (
            -row["score"],
            row["category_rank"],
            row["list_position"],
        ),
    )


def select_attribute_shortlist(
    query,
    category_id,
    attributes,
    shortlist_size=None,
):
    """
    Выбирает labels для GLiNER2.

    shortlist_size=None означает полную production-схему.
    """
    attributes = list(dict.fromkeys(attributes or []))

    if shortlist_size is None or shortlist_size >= len(attributes):
        return attributes, [
            {
                "attribute": attribute,
                "score": None,
                "category_rank": position,
                "list_position": position,
                "reasons": ["full_schema"],
            }
            for position, attribute in enumerate(attributes, start=1)
        ]

    shortlist_size = int(shortlist_size)

    if shortlist_size <= 0:
        raise ValueError("shortlist_size должен быть положительным или None")

    ranking = rank_attributes_for_query(
        query=query,
        category_id=category_id,
        attributes=attributes,
    )

    selected_set = {
        row["attribute"]
        for row in ranking[:shortlist_size]
    }

    # Передаём labels в исходном стабильном порядке production-схемы.
    selected = [
        attribute
        for attribute in attributes
        if attribute in selected_set
    ]

    return selected, ranking


# Быстрая визуальная проверка selector.
SHORTLIST_EXAMPLE_QUERY = "Samsung Galaxy S26 256 ГБ оперативы"
SHORTLIST_EXAMPLE_CATEGORY = "16434"

if SHORTLIST_EXAMPLE_CATEGORY in category_attributes:
    for size in [5, 10]:
        selected, ranking = select_attribute_shortlist(
            SHORTLIST_EXAMPLE_QUERY,
            SHORTLIST_EXAMPLE_CATEGORY,
            category_attributes[SHORTLIST_EXAMPLE_CATEGORY],
            shortlist_size=size,
        )
        print(f"shortlist-{size}:", selected)
        display(pd.DataFrame(ranking[:size]))


### Финальная функция

Функция `predict_product_attributes` выполняет весь пайплайн:

1. нормализует запрос;
2. предсказывает категорию;
3. получает список атрибутов;
4. запускает GLiNER2;
5. возвращает категорию, уверенность, факты и время выполнения.


In [ ]:
def _synchronize_cuda_for_latency():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def predict_product_attributes(
    query,
    gliner_threshold=0.20,
    category_top_k=3,
    shortlist_size=5,
):
    """Извлекает товарные атрибуты из одного поискового запроса."""
    if query is None or not str(query).strip():
        raise ValueError("Запрос не должен быть пустым.")

    _synchronize_cuda_for_latency()
    started_at = time.perf_counter()

    category_predictions = predict_category(
        query,
        k=category_top_k,
    )

    top_prediction = category_predictions[0]
    category_id = top_prediction["category_id"]

    all_attributes = category_attributes.get(category_id)
    used_fallback_schema = not bool(all_attributes)

    if not all_attributes:
        all_attributes = GENERIC_ATTRIBUTES.copy()

    attributes, shortlist_ranking = select_attribute_shortlist(
        query=query,
        category_id=category_id,
        attributes=all_attributes,
        shortlist_size=shortlist_size,
    )

    raw_result = extractor.extract_entities(
        str(query),
        attributes,
        threshold=gliner_threshold,
        include_confidence=True,
        include_spans=True,
    )

    if isinstance(raw_result, dict):
        facts = raw_result.get(
            "entities",
            raw_result,
        )
    else:
        facts = {}

    _synchronize_cuda_for_latency()
    latency_ms = (
        time.perf_counter() - started_at
    ) * 1_000

    category_name = category_meta.get(
        category_id,
        {},
    ).get("category_name")

    return {
        "query": str(query),
        "normalized_query": normalize_query_for_fasttext(query),
        "category_id": category_id,
        "category_name": category_name,
        "category_confidence": top_prediction["confidence"],
        "category_top_k": category_predictions,
        "used_fallback_schema": used_fallback_schema,
        "all_candidate_attributes": all_attributes,
        "candidate_attributes": attributes,
        "n_all_candidate_attributes": len(all_attributes),
        "n_candidate_attributes": len(attributes),
        "shortlist_size": shortlist_size,
        "shortlist_ranking": shortlist_ranking,
        "facts": facts,
        "flat_facts": flatten_facts(facts),
        "latency_ms": latency_ms,
    }


### Примеры работы

Набор содержит запросы с разным порядком слов и товарами из нескольких категорий. Результат выводится в компактном виде, без внутреннего служебного ответа GLiNER2.


In [ ]:
test_queries = [
    "Samsung Galaxy S26 256 ГБ черный",
    "Galaxy Samsung S26 256 ГБ",
    "айфон 16 про макс 512 гб",
    "телевизор lg oled 55 дюймов",
    "красные кроссовки nike 42 размер",
]

for query in test_queries:
    result = predict_product_attributes(query)

    print("=" * 100)
    print("Запрос:", result["query"])
    print(
        "Категория:",
        result["category_id"],
        result["category_name"] or "",
    )
    print(
        "Уверенность:",
        f"{result['category_confidence']:.4f}",
    )
    print(
        "Fallback-схема:",
        result["used_fallback_schema"],
    )
    print(
        "Время:",
        f"{result['latency_ms']:.2f} мс",
    )
    print("Факты:")

    if result["flat_facts"]:
        display(pd.DataFrame(result["flat_facts"]))
    else:
        print("Факты не найдены.")


На Tesla T4 финальный вариант `base GLiNER2 + shortlist-5` показал:

- p50: **27,0 мс**;
- p95: **40,4 мс**.

Таким образом, ограничение 120 мс достигается на GPU. Замеры на TPU-runtime без PyTorch/XLA фактически выполнялись на CPU и не используются как итоговые.


## 10. Базовая проверка инференса

Это не полноценная оценка качества NER, поскольку в исходных данных нет ручной разметки границ и атрибутов. Ячейка измеряет технические характеристики baseline:

- долю ошибок;
- долю запросов без фактов;
- среднее количество извлечённых фактов;
- распределение latency.

Для быстрой проверки используется случайная выборка уникальных запросов.


In [ ]:
N_EVALUATION_QUERIES = 200
GLINER_THRESHOLD = 0.30

unique_queries = (
    query_clicks[["query_text"]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

evaluation_queries = unique_queries.sample(
    n=min(N_EVALUATION_QUERIES, len(unique_queries)),
    random_state=RANDOM_STATE,
).reset_index(drop=True)

evaluation_results = []

for query in tqdm(
    evaluation_queries["query_text"],
    desc="Проверка инференса",
):
    try:
        result = predict_product_attributes(
            query,
            gliner_threshold=GLINER_THRESHOLD,
        )

        evaluation_results.append(
            {
                "query": query,
                "category_id": result["category_id"],
                "category_confidence": (
                    result["category_confidence"]
                ),
                "used_fallback_schema": (
                    result["used_fallback_schema"]
                ),
                "n_candidate_attributes": (
                    result["n_candidate_attributes"]
                ),
                "n_extracted_facts": len(
                    result["flat_facts"]
                ),
                "latency_ms": result["latency_ms"],
                "error": None,
            }
        )

    except Exception as exception:
        evaluation_results.append(
            {
                "query": query,
                "category_id": None,
                "category_confidence": None,
                "used_fallback_schema": None,
                "n_candidate_attributes": None,
                "n_extracted_facts": 0,
                "latency_ms": None,
                "error": repr(exception),
            }
        )

baseline_results = pd.DataFrame(evaluation_results)

print(
    "Ошибок:",
    int(baseline_results["error"].notna().sum()),
)
print(
    "Доля запросов без фактов:",
    f"{(baseline_results['n_extracted_facts'] == 0).mean():.2%}",
)
print(
    "Среднее число фактов:",
    f"{baseline_results['n_extracted_facts'].mean():.2f}",
)
print(
    "Доля fallback-схем:",
    f"{baseline_results['used_fallback_schema'].fillna(False).mean():.2%}",
)

display(
    baseline_results[
        [
            "category_confidence",
            "n_candidate_attributes",
            "n_extracted_facts",
            "latency_ms",
        ]
    ].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    )
)


## 11. Упаковка локальных артефактов

Артефакты не следует коммитить в публичный GitHub-репозиторий, если они получены из приватных данных. Для локального переноса их можно упаковать в ZIP или хранить в GitHub Releases / Git LFS.


In [ ]:
BUNDLE_PATH = PROJECT_ROOT / "marketplace_attribute_extractor_artifacts.zip"

required_files_to_pack = [
    MODEL_PATH,
    METRICS_PATH,
    ATTRIBUTE_STATS_PATH,
    ATTRIBUTE_CANDIDATES_PATH,
    CATEGORY_ATTRIBUTES_PATH,
    CATEGORY_META_PATH,
]

optional_files_to_pack = [
    GLINER_DATASET_REPORT_PATH,
    GLINER_TRAINING_REPORT_PATH,
    GLINER_COMPARISON_PATH,
]

missing_artifacts = [
    path
    for path in required_files_to_pack
    if not path.exists()
]

if missing_artifacts:
    raise FileNotFoundError(
        "Не найдены артефакты: "
        + ", ".join(str(path) for path in missing_artifacts)
    )

with zipfile.ZipFile(
    BUNDLE_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for artifact_path in required_files_to_pack:
        archive.write(
            artifact_path,
            arcname=artifact_path.name,
        )

    for artifact_path in optional_files_to_pack:
        if artifact_path.exists():
            archive.write(
                artifact_path,
                arcname=artifact_path.name,
            )

    if GLINER_ADAPTER_FINAL_DIR.exists():
        for adapter_file in GLINER_ADAPTER_FINAL_DIR.rglob("*"):
            if adapter_file.is_file():
                archive.write(
                    adapter_file,
                    arcname=str(
                        Path("gliner2_marketplace_lora")
                        / "final"
                        / adapter_file.relative_to(
                            GLINER_ADAPTER_FINAL_DIR
                        )
                    ),
                )

print("Архив создан:", BUNDLE_PATH)
print(
    "Размер:",
    f"{BUNDLE_PATH.stat().st_size / 1024**2:.2f} MB",
)


## Промежуточный итог

Получен end-to-end пайплайн:

```text
fastText -> top-1 категория -> dynamic shortlist-5 -> base GLiNER2
```

Главный инженерный результат — сокращение пространства labels улучшило не только скорость, но и качество. Более сложный LoRA-вариант не был выбран, поскольку уступил baseline на отложенной оценке.


## 12. Оценка на proxy-golden датасете

Набор содержит 1 000 синтетически размеченных примеров с частичной ручной проверкой. Он используется для контролируемого сравнения вариантов, но не позиционируется как полностью human-labeled production benchmark.


Формат expected facts допускает неканонические названия из пользовательских запросов. Метрики используют нормализацию поверхности и отдельно оценивают span/value и корректность выбранного атрибута.


In [ ]:
GOLDEN_PATH = Path(
    os.environ.get(
        "GOLDEN_PATH",
        PROJECT_ROOT / "data" / "proxy_golden_1000.jsonl",
    )
)

if not GOLDEN_PATH.exists():
    raise FileNotFoundError(
        f"Не найден proxy-golden файл: {GOLDEN_PATH}. "
        "Он не включён в публичный репозиторий."
    )


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

golden_df = pd.read_json(
    GOLDEN_PATH,
    lines=True,
)

golden_df["expected_category_id"] = (
    golden_df["expected_category_id"].astype(str)
)

print(f"Всего примеров: {len(golden_df):,}")
print("\nРазбиение:")
display(golden_df["split"].value_counts().to_frame("rows"))

print("\nСложность:")
display(golden_df["difficulty"].value_counts().to_frame("rows"))

golden_df.head(3)

In [ ]:
# assert "predict_product_attributes" in globals(), (
#     "Сначала запусти ячейку с функцией "
#     "predict_product_attributes."
# )

proxy_dev = (
    golden_df[golden_df["split"] == "proxy_dev"]
    .copy()
    .reset_index(drop=True)
)

proxy_test = (
    golden_df[golden_df["split"] == "proxy_test"]
    .copy()
    .reset_index(drop=True)
)

print(f"proxy_dev:  {len(proxy_dev):,}")
print(f"proxy_test: {len(proxy_test):,}")

#### Функции для запуска моделей

In [ ]:
from tqdm.auto import tqdm


def run_golden_evaluation(
    dataset,
    gliner_threshold=0.30,
    category_top_k=3,
    shortlist_size=None,
    progress_description="Проверка модели",
    fail_on_error=True,
):
    """
    Последовательно запускает полный пайплайн:
    fastText -> выбор схемы категории -> shortlist -> GLiNER2.
    """
    result_rows = []

    records = dataset.to_dict("records")

    for row in tqdm(records, desc=progress_description):
        try:
            prediction = predict_product_attributes(
                query=row["query_text"],
                gliner_threshold=gliner_threshold,
                category_top_k=category_top_k,
                shortlist_size=shortlist_size,
            )

            category_top_k_predictions = [
                str(item["category_id"])
                for item in prediction.get("category_top_k", [])
            ]

            predicted_facts = []

            for fact in prediction.get("flat_facts", []):
                predicted_facts.append(
                    {
                        "attribute": fact.get("attribute"),
                        "value": fact.get("value"),
                        "confidence": fact.get("confidence"),
                        "start": fact.get("start"),
                        "end": fact.get("end"),
                    }
                )

            result_rows.append(
                {
                    **row,
                    "predicted_category_id": str(
                        prediction.get("category_id", "")
                    ),
                    "predicted_category_name": prediction.get(
                        "category_name"
                    ),
                    "predicted_category_confidence": prediction.get(
                        "category_confidence"
                    ),
                    "predicted_category_top_k": (
                        category_top_k_predictions
                    ),
                    "predicted_facts": predicted_facts,
                    "predicted_fact_count": len(predicted_facts),
                    "used_fallback_schema": prediction.get(
                        "used_fallback_schema"
                    ),
                    "all_candidate_attributes": prediction.get(
                        "all_candidate_attributes", []
                    ),
                    "candidate_attributes": prediction.get(
                        "candidate_attributes", []
                    ),
                    "n_all_candidate_attributes": prediction.get(
                        "n_all_candidate_attributes"
                    ),
                    "n_candidate_attributes": prediction.get(
                        "n_candidate_attributes"
                    ),
                    "shortlist_size": shortlist_size,
                    "latency_ms": prediction.get("latency_ms"),
                    "execution_error": None,
                }
            )

        except Exception as error:
            if fail_on_error:
                raise RuntimeError(
                    f"Ошибка инференса для запроса {row.get('query_text')!r}"
                ) from error
            result_rows.append(
                {
                    **row,
                    "predicted_category_id": "",
                    "predicted_category_name": None,
                    "predicted_category_confidence": np.nan,
                    "predicted_category_top_k": [],
                    "predicted_facts": [],
                    "predicted_fact_count": 0,
                    "used_fallback_schema": None,
                    "all_candidate_attributes": [],
                    "candidate_attributes": [],
                    "n_all_candidate_attributes": np.nan,
                    "n_candidate_attributes": np.nan,
                    "shortlist_size": shortlist_size,
                    "latency_ms": np.nan,
                    "execution_error": (
                        f"{type(error).__name__}: {error}"
                    ),
                }
            )

    return pd.DataFrame(result_rows)


#### Функции для рассчета метрик

In [ ]:
import re
import unicodedata


def normalize_surface_value(value):
    """
    Базовая нормализация поверхности значения.
    Не меняет смысл значения и не использует словари товаров.
    """
    if value is None:
        return ""

    value = unicodedata.normalize("NFKC", str(value))
    value = value.lower().replace("ё", "е")

    # Разделяем число и единицу измерения.
    value = re.sub(
        r"(?<=\d)(гб|мб|тб|мач|вт|квт|гц|кг|см|мм|л)\b",
        r" \1",
        value,
    )

    value = re.sub(r"\s+", " ", value).strip()
    return value


def facts_to_set(facts, comparison_mode="span"):
    """
    comparison_mode:
    - span: атрибут + точные координаты;
    - surface: атрибут + текст значения;
    - attribute: только название атрибута.
    """
    result = set()

    for fact in facts or []:
        attribute = str(fact.get("attribute", "")).strip()

        if not attribute:
            continue

        if comparison_mode == "span":
            start = fact.get("start")
            end = fact.get("end")

            if start is None or end is None:
                continue

            result.add(
                (
                    attribute,
                    int(start),
                    int(end),
                )
            )

        elif comparison_mode == "surface":
            value = normalize_surface_value(
                fact.get("value")
            )

            if value:
                result.add((attribute, value))

        elif comparison_mode == "attribute":
            result.add(attribute)

        else:
            raise ValueError(
                f"Неизвестный comparison_mode: {comparison_mode}"
            )

    return result


def filter_facts_by_threshold(facts, threshold):
    """Фильтрует уже полученные факты по confidence."""
    filtered = []

    for fact in facts or []:
        confidence = fact.get("confidence")

        if confidence is None:
            filtered.append(fact)
        elif float(confidence) >= threshold:
            filtered.append(fact)

    return filtered


def calculate_micro_prf(
    evaluation_df,
    comparison_mode,
    threshold=None,
):
    true_positive = 0
    false_positive = 0
    false_negative = 0
    exact_matches = 0

    evaluated_rows = 0

    for _, row in evaluation_df.iterrows():
        if not bool(row["ner_evaluable"]):
            continue

        gold_facts = row["expected_facts"]
        predicted_facts = row["predicted_facts"]

        if threshold is not None:
            predicted_facts = filter_facts_by_threshold(
                predicted_facts,
                threshold,
            )

        gold_set = facts_to_set(
            gold_facts,
            comparison_mode=comparison_mode,
        )

        predicted_set = facts_to_set(
            predicted_facts,
            comparison_mode=comparison_mode,
        )

        true_positive += len(gold_set & predicted_set)
        false_positive += len(predicted_set - gold_set)
        false_negative += len(gold_set - predicted_set)

        exact_matches += int(gold_set == predicted_set)
        evaluated_rows += 1

    precision_denominator = (
        true_positive + false_positive
    )
    recall_denominator = (
        true_positive + false_negative
    )

    precision = (
        true_positive / precision_denominator
        if precision_denominator
        else 0.0
    )

    recall = (
        true_positive / recall_denominator
        if recall_denominator
        else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    exact_match = (
        exact_matches / evaluated_rows
        if evaluated_rows
        else np.nan
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "exact_match": exact_match,
        "tp": true_positive,
        "fp": false_positive,
        "fn": false_negative,
    }


def calculate_all_metrics(
    evaluation_df,
    threshold=None,
):
    category_rows = evaluation_df[
        evaluation_df["category_evaluable"] == True
    ].copy()

    category_accuracy = (
        category_rows["predicted_category_id"]
        == category_rows["expected_category_id"]
    ).mean()

    category_recall_at_3 = np.mean(
        [
            expected_id in predicted_ids
            for expected_id, predicted_ids in zip(
                category_rows["expected_category_id"],
                category_rows["predicted_category_top_k"],
            )
        ]
    )

    span_metrics = calculate_micro_prf(
        evaluation_df,
        comparison_mode="span",
        threshold=threshold,
    )

    surface_metrics = calculate_micro_prf(
        evaluation_df,
        comparison_mode="surface",
        threshold=threshold,
    )

    attribute_metrics = calculate_micro_prf(
        evaluation_df,
        comparison_mode="attribute",
        threshold=threshold,
    )

    negative_rows = evaluation_df[
        (evaluation_df["ner_evaluable"] == True)
        & (evaluation_df["expected_fact_count"] == 0)
    ]

    positive_rows = evaluation_df[
        (evaluation_df["ner_evaluable"] == True)
        & (evaluation_df["expected_fact_count"] > 0)
    ]

    if threshold is None:
        negative_predicted_counts = (
            negative_rows["predicted_fact_count"]
        )

        positive_predicted_counts = (
            positive_rows["predicted_fact_count"]
        )
    else:
        negative_predicted_counts = negative_rows[
            "predicted_facts"
        ].apply(
            lambda facts: len(
                filter_facts_by_threshold(facts, threshold)
            )
        )

        positive_predicted_counts = positive_rows[
            "predicted_facts"
        ].apply(
            lambda facts: len(
                filter_facts_by_threshold(facts, threshold)
            )
        )

    hallucination_rate = (
        (negative_predicted_counts > 0).mean()
        if len(negative_rows)
        else np.nan
    )

    missed_all_facts_rate = (
        (positive_predicted_counts == 0).mean()
        if len(positive_rows)
        else np.nan
    )

    latency = (
        pd.to_numeric(
            evaluation_df["latency_ms"],
            errors="coerce",
        )
        .dropna()
    )

    return {
        "rows": len(evaluation_df),
        "execution_errors": int(
            evaluation_df["execution_error"]
            .notna()
            .sum()
        ),
        "category_accuracy_at_1": category_accuracy,
        "category_recall_at_3": category_recall_at_3,
        "span_precision": span_metrics["precision"],
        "span_recall": span_metrics["recall"],
        "span_f1": span_metrics["f1"],
        "span_exact_match": span_metrics["exact_match"],
        "surface_f1": surface_metrics["f1"],
        "surface_exact_match": surface_metrics[
            "exact_match"
        ],
        "attribute_f1": attribute_metrics["f1"],
        "hallucination_rate": hallucination_rate,
        "missed_all_facts_rate": missed_all_facts_rate,
        "latency_mean_ms": latency.mean(),
        "latency_p50_ms": latency.quantile(0.50),
        "latency_p95_ms": latency.quantile(0.95),
        "latency_p99_ms": latency.quantile(0.99),
    }

#### Настройка порога на proxy_dev

Модель запускается с низким порогом 0.10, после чего мы проверяем несколько порогов без повторного инференса.

In [ ]:
dev_predictions = run_golden_evaluation(
    proxy_dev,
    gliner_threshold=0.10,
    category_top_k=3,
)

threshold_rows = []

for threshold in np.arange(0.10, 0.71, 0.05):
    threshold = round(float(threshold), 2)

    metrics = calculate_all_metrics(
        dev_predictions,
        threshold=threshold,
    )

    threshold_rows.append(
        {
            "threshold": threshold,
            "span_precision": metrics["span_precision"],
            "span_recall": metrics["span_recall"],
            "span_f1": metrics["span_f1"],
            "surface_f1": metrics["surface_f1"],
            "attribute_f1": metrics["attribute_f1"],
            "hallucination_rate": metrics[
                "hallucination_rate"
            ],
            "span_exact_match": metrics[
                "span_exact_match"
            ],
        }
    )

threshold_results = pd.DataFrame(threshold_rows)

threshold_results = threshold_results.sort_values(
    [
        "span_f1",
        "span_precision",
        "span_exact_match",
    ],
    ascending=False,
).reset_index(drop=True)

BEST_GLINER_THRESHOLD = float(
    threshold_results.iloc[0]["threshold"]
)

print(
    "Лучший порог GLiNER2:",
    BEST_GLINER_THRESHOLD,
)

display(
    threshold_results.style.format(
        {
            "span_precision": "{:.4f}",
            "span_recall": "{:.4f}",
            "span_f1": "{:.4f}",
            "surface_f1": "{:.4f}",
            "attribute_f1": "{:.4f}",
            "hallucination_rate": "{:.4f}",
            "span_exact_match": "{:.4f}",
        }
    )
)

#### Финальная проверка на proxy_test

In [ ]:
test_predictions = run_golden_evaluation(
    proxy_test,
    gliner_threshold=BEST_GLINER_THRESHOLD,
    category_top_k=3,
)

test_metrics = calculate_all_metrics(
    test_predictions
)

metrics_table = pd.DataFrame(
    {
        "Метрика": [
            "Количество примеров",
            "Ошибки выполнения",
            "Category Accuracy@1",
            "Category Recall@3",
            "Span Precision",
            "Span Recall",
            "Span F1",
            "Строгий Entity Exact Match",
            "Surface F1",
            "Attribute F1",
            "Доля галлюцинаций",
            "Доля полных пропусков",
            "Средняя задержка, мс",
            "P50 задержки, мс",
            "P95 задержки, мс",
            "P99 задержки, мс",
        ],
        "Значение": [
            test_metrics["rows"],
            test_metrics["execution_errors"],
            test_metrics["category_accuracy_at_1"],
            test_metrics["category_recall_at_3"],
            test_metrics["span_precision"],
            test_metrics["span_recall"],
            test_metrics["span_f1"],
            test_metrics["span_exact_match"],
            test_metrics["surface_f1"],
            test_metrics["attribute_f1"],
            test_metrics["hallucination_rate"],
            test_metrics["missed_all_facts_rate"],
            test_metrics["latency_mean_ms"],
            test_metrics["latency_p50_ms"],
            test_metrics["latency_p95_ms"],
            test_metrics["latency_p99_ms"],
        ],
    }
)

display(
    metrics_table.style.format(
        {
            "Значение": lambda value: (
                f"{value:.4f}"
                if isinstance(value, (float, np.floating))
                else str(value)
            )
        }
    )
)

#### Метрики по сложности запросов

In [ ]:
difficulty_reports = []

for difficulty, subset in test_predictions.groupby(
    "difficulty"
):
    metrics = calculate_all_metrics(subset)

    difficulty_reports.append(
        {
            "difficulty": difficulty,
            "rows": len(subset),
            "category_accuracy_at_1": metrics[
                "category_accuracy_at_1"
            ],
            "category_recall_at_3": metrics[
                "category_recall_at_3"
            ],
            "span_precision": metrics["span_precision"],
            "span_recall": metrics["span_recall"],
            "span_f1": metrics["span_f1"],
            "surface_f1": metrics["surface_f1"],
            "attribute_f1": metrics["attribute_f1"],
            "hallucination_rate": metrics[
                "hallucination_rate"
            ],
            "latency_p95_ms": metrics["latency_p95_ms"],
        }
    )

difficulty_report = pd.DataFrame(
    difficulty_reports
).sort_values(
    "difficulty",
    key=lambda column: column.map(
        {
            "easy": 0,
            "medium": 1,
            "hard": 2,
        }
    ),
)

display(
    difficulty_report.style.format(
        {
            column: "{:.4f}"
            for column in difficulty_report.columns
            if column not in {"difficulty", "rows"}
        }
    )
)

#### Метрики по категориям

In [ ]:
category_reports = []

for (
    expected_category_id,
    category_name,
), subset in test_predictions.groupby(
    [
        "expected_category_id",
        "category_name",
    ],
    dropna=False,
):
    metrics = calculate_all_metrics(subset)

    category_reports.append(
        {
            "category_id": expected_category_id,
            "category_name": category_name,
            "rows": len(subset),
            "category_accuracy_at_1": metrics[
                "category_accuracy_at_1"
            ],
            "category_recall_at_3": metrics[
                "category_recall_at_3"
            ],
            "span_f1": metrics["span_f1"],
            "surface_f1": metrics["surface_f1"],
            "attribute_f1": metrics["attribute_f1"],
            "hallucination_rate": metrics[
                "hallucination_rate"
            ],
            "latency_p95_ms": metrics["latency_p95_ms"],
        }
    )

category_report = (
    pd.DataFrame(category_reports)
    .sort_values(
        [
            "category_accuracy_at_1",
            "span_f1",
        ]
    )
    .reset_index(drop=True)
)

display(
    category_report.style.format(
        {
            "category_accuracy_at_1": "{:.4f}",
            "category_recall_at_3": "{:.4f}",
            "span_f1": "{:.4f}",
            "surface_f1": "{:.4f}",
            "attribute_f1": "{:.4f}",
            "hallucination_rate": "{:.4f}",
            "latency_p95_ms": "{:.2f}",
        }
    )
)

#### Просмотр ошибок

In [ ]:
def format_facts(facts):
    if not facts:
        return "—"

    formatted = []

    for fact in facts:
        attribute = fact.get("attribute")
        value = fact.get("value")
        confidence = fact.get("confidence")

        if confidence is None:
            formatted.append(
                f"{attribute}: {value}"
            )
        else:
            formatted.append(
                f"{attribute}: {value} ({confidence:.3f})"
            )

    return " | ".join(formatted)


error_rows = []

for _, row in test_predictions.iterrows():
    gold_span_set = facts_to_set(
        row["expected_facts"],
        comparison_mode="span",
    )

    predicted_span_set = facts_to_set(
        row["predicted_facts"],
        comparison_mode="span",
    )

    category_is_correct = (
        row["expected_category_id"]
        == row["predicted_category_id"]
    )

    facts_are_correct = (
        gold_span_set == predicted_span_set
    )

    if not category_is_correct or not facts_are_correct:
        error_rows.append(
            {
                "example_id": row["example_id"],
                "query_text": row["query_text"],
                "difficulty": row["difficulty"],
                "expected_category": (
                    row["expected_category_id"]
                ),
                "predicted_category": (
                    row["predicted_category_id"]
                ),
                "category_correct": category_is_correct,
                "expected_facts": format_facts(
                    row["expected_facts"]
                ),
                "predicted_facts": format_facts(
                    row["predicted_facts"]
                ),
                "perturbation_type": row[
                    "perturbation_type"
                ],
                "latency_ms": row["latency_ms"],
                "execution_error": row["execution_error"],
            }
        )

errors_df = pd.DataFrame(error_rows)

print(
    f"Ошибочных примеров: "
    f"{len(errors_df):,} из {len(test_predictions):,}"
)

display(
    errors_df.head(100).style.format(
        {
            "latency_ms": "{:.2f}",
        }
    )
)

## 13. Эксперимент shortlist: полная схема против 5 и 10 атрибутов

Необходимо понять, сколько атрибутов передавать GLiNER-у для лучшего качества. Для этого сравним качества модели, если ей передавать топ 5/10 самых частых атрибутов.

Для каждого режима threshold подбирался отдельно на `proxy_dev`, после чего метрики считались на `proxy_test`.

Итог:

| Режим | Span F1 | Attribute F1 |
|---|---:|---:|
| Полная схема | 0,6335 | 0,6653 |
| **Shortlist-5** | **0,7035** | **0,7674** |
| Shortlist-10 | 0,6358 | 0,6732 |


In [ ]:
RUN_SHORTLIST_COMPARISON = False
SHORTLIST_MODES = [None, 5, 10]
SHORTLIST_COMPARISON_PATH = RESULTS_DIR / "shortlist_comparison.csv"
SHORTLIST_LATENCY_PATH = RESULTS_DIR / "shortlist_latency.csv"

LATENCY_BENCHMARK_SAMPLE_SIZE = 40
LATENCY_BENCHMARK_REPEATS = 3


def mode_name(shortlist_size):
    return (
        "full"
        if shortlist_size is None
        else f"shortlist_{int(shortlist_size)}"
    )


def calculate_shortlist_coverage(evaluation_df):
    matched_attributes = 0
    total_attributes = 0
    fully_covered_examples = 0
    evaluated_examples = 0

    for row in evaluation_df.to_dict("records"):
        if not bool(row.get("ner_evaluable")):
            continue

        gold_attributes = {
            str(fact.get("attribute", "")).strip()
            for fact in row.get("expected_facts", [])
            if str(fact.get("attribute", "")).strip()
        }

        if not gold_attributes:
            continue

        candidate_attributes = set(
            row.get("candidate_attributes", []) or []
        )

        matched_attributes += len(
            gold_attributes & candidate_attributes
        )
        total_attributes += len(gold_attributes)
        fully_covered_examples += int(
            gold_attributes <= candidate_attributes
        )
        evaluated_examples += 1

    return {
        "shortlist_attribute_recall": (
            matched_attributes / total_attributes
            if total_attributes
            else np.nan
        ),
        "shortlist_full_coverage": (
            fully_covered_examples / evaluated_examples
            if evaluated_examples
            else np.nan
        ),
        "shortlist_evaluated_examples": evaluated_examples,
    }


def tune_shortlist_threshold(dev_predictions):
    rows = []

    for threshold in np.arange(0.10, 0.71, 0.05):
        threshold = round(float(threshold), 2)
        metrics = calculate_all_metrics(
            dev_predictions,
            threshold=threshold,
        )
        rows.append(
            {
                "threshold": threshold,
                "span_precision": metrics["span_precision"],
                "span_recall": metrics["span_recall"],
                "span_f1": metrics["span_f1"],
                "surface_f1": metrics["surface_f1"],
                "attribute_f1": metrics["attribute_f1"],
                "hallucination_rate": metrics[
                    "hallucination_rate"
                ],
            }
        )

    threshold_table = pd.DataFrame(rows).sort_values(
        ["span_f1", "span_precision", "surface_f1"],
        ascending=False,
    ).reset_index(drop=True)

    return (
        float(threshold_table.iloc[0]["threshold"]),
        threshold_table,
    )


def warmup_shortlist_mode(shortlist_size, n_queries=5):
    warmup_queries = (
        proxy_dev["query_text"]
        .dropna()
        .astype(str)
        .head(n_queries)
        .tolist()
    )

    for query in warmup_queries:
        predict_product_attributes(
            query=query,
            gliner_threshold=0.10,
            category_top_k=3,
            shortlist_size=shortlist_size,
        )


def benchmark_shortlist_latency(
    dataset,
    shortlist_size,
    threshold,
    sample_size=40,
    repeats=3,
):
    benchmark_frame = (
        dataset
        .drop_duplicates("query_text")
        .head(sample_size)
        .copy()
    )

    latency_values = []

    warmup_shortlist_mode(shortlist_size)

    for repeat_index in range(repeats):
        predictions = run_golden_evaluation(
            benchmark_frame,
            gliner_threshold=threshold,
            category_top_k=3,
            shortlist_size=shortlist_size,
            progress_description=(
                f"Latency {mode_name(shortlist_size)} "
                f"{repeat_index + 1}/{repeats}"
            ),
        )

        latency_values.extend(
            pd.to_numeric(
                predictions["latency_ms"],
                errors="coerce",
            )
            .dropna()
            .tolist()
        )

    latency_values = pd.Series(
        latency_values,
        dtype="float64",
    )

    return {
        "latency_benchmark_rows": len(latency_values),
        "benchmark_latency_mean_ms": latency_values.mean(),
        "benchmark_latency_p50_ms": latency_values.quantile(0.50),
        "benchmark_latency_p95_ms": latency_values.quantile(0.95),
        "benchmark_latency_p99_ms": latency_values.quantile(0.99),
    }


if RUN_SHORTLIST_COMPARISON:
    shortlist_comparison_rows = []
    shortlist_threshold_tables = {}
    shortlist_latency_rows = []

    for shortlist_size in SHORTLIST_MODES:
        current_mode = mode_name(shortlist_size)
        print("\nРежим:", current_mode)

        warmup_shortlist_mode(shortlist_size)

        dev_result = run_golden_evaluation(
            proxy_dev,
            gliner_threshold=0.10,
            category_top_k=3,
            shortlist_size=shortlist_size,
            progress_description=f"Dev {current_mode}",
        )

        best_threshold, threshold_table = (
            tune_shortlist_threshold(dev_result)
        )
        shortlist_threshold_tables[current_mode] = threshold_table

        test_result = run_golden_evaluation(
            proxy_test,
            gliner_threshold=best_threshold,
            category_top_k=3,
            shortlist_size=shortlist_size,
            progress_description=f"Test {current_mode}",
        )

        metrics = calculate_all_metrics(test_result)
        coverage = calculate_shortlist_coverage(test_result)

        comparison_row = {
            "mode": current_mode,
            "shortlist_size": (
                np.nan if shortlist_size is None else shortlist_size
            ),
            "best_threshold": best_threshold,
            "mean_candidate_attributes": pd.to_numeric(
                test_result["n_candidate_attributes"],
                errors="coerce",
            ).mean(),
            **coverage,
            **metrics,
        }
        shortlist_comparison_rows.append(comparison_row)

        latency_row = {
            "mode": current_mode,
            **benchmark_shortlist_latency(
                dataset=proxy_test,
                shortlist_size=shortlist_size,
                threshold=best_threshold,
                sample_size=LATENCY_BENCHMARK_SAMPLE_SIZE,
                repeats=LATENCY_BENCHMARK_REPEATS,
            ),
        }
        shortlist_latency_rows.append(latency_row)

        del dev_result
        del test_result
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    shortlist_comparison = pd.DataFrame(
        shortlist_comparison_rows
    )
    shortlist_latency = pd.DataFrame(
        shortlist_latency_rows
    )

    shortlist_comparison = shortlist_comparison.merge(
        shortlist_latency,
        on="mode",
        how="left",
    )

    shortlist_comparison.to_csv(
        SHORTLIST_COMPARISON_PATH,
        index=False,
    )
    shortlist_latency.to_csv(
        SHORTLIST_LATENCY_PATH,
        index=False,
    )

    display_columns = [
        "mode",
        "mean_candidate_attributes",
        "best_threshold",
        "shortlist_attribute_recall",
        "shortlist_full_coverage",
        "span_precision",
        "span_recall",
        "span_f1",
        "surface_f1",
        "attribute_f1",
        "hallucination_rate",
        "benchmark_latency_p50_ms",
        "benchmark_latency_p95_ms",
    ]

    display(
        shortlist_comparison[display_columns]
        .sort_values("benchmark_latency_p95_ms")
        .style.format(
            {
                column: "{:.4f}"
                for column in [
                    "best_threshold",
                    "shortlist_attribute_recall",
                    "shortlist_full_coverage",
                    "span_precision",
                    "span_recall",
                    "span_f1",
                    "surface_f1",
                    "attribute_f1",
                    "hallucination_rate",
                ]
            }
            | {
                "mean_candidate_attributes": "{:.2f}",
                "benchmark_latency_p50_ms": "{:.2f}",
                "benchmark_latency_p95_ms": "{:.2f}",
            }
        )
    )

    # Рекомендация: сначала соблюдение latency, затем качество.
    eligible = shortlist_comparison[
        shortlist_comparison["benchmark_latency_p95_ms"] <= 120.0
    ].copy()

    if not eligible.empty:
        recommended = eligible.sort_values(
            [
                "span_f1",
                "shortlist_attribute_recall",
                "benchmark_latency_p95_ms",
            ],
            ascending=[False, False, True],
        ).iloc[0]
        recommendation_reason = "p95 <= 120 мс"
    else:
        recommended = shortlist_comparison.sort_values(
            [
                "benchmark_latency_p95_ms",
                "span_f1",
                "shortlist_attribute_recall",
            ],
            ascending=[True, False, False],
        ).iloc[0]
        recommendation_reason = (
            "ни один режим пока не достиг p95 <= 120 мс"
        )

    print()
    print("Рекомендуемый режим:", recommended["mode"])
    print("Причина:", recommendation_reason)
    print(
        "Span F1:",
        f"{recommended['span_f1']:.4f}",
    )
    print(
        "Shortlist attribute recall:",
        f"{recommended['shortlist_attribute_recall']:.4f}",
    )
    print(
        "Latency p95:",
        f"{recommended['benchmark_latency_p95_ms']:.2f} мс",
    )
    print("Сравнение сохранено:", SHORTLIST_COMPARISON_PATH)
    print("Latency сохранена:", SHORTLIST_LATENCY_PATH)


### Вывод по shortlist

Shortlist-5 выбран как финальный режим:

- Span F1 вырос на **7,0 п.п.** относительно полной схемы;
- Attribute F1 вырос на **10,2 п.п.**;
- число labels сокращено примерно с 25 до 5;
- на Tesla T4 end-to-end p95 составил **40,4 мс**.

Рост hallucination rate связан с низким оптимальным threshold (`0.2`) и учитывается как ограничение proxy-golden оценки.


## 14. Сравнение базового GLiNER2 и LoRA

Для каждой версии threshold подбирался отдельно на `proxy_dev`, итоговые метрики считались на `proxy_test`.

| Вариант | Threshold | Span F1 | Attribute F1 | p95 на T4 |
|---|---:|---:|---:|---:|
| **Base GLiNER2** | 0,2 | **0,7035** | **0,7674** | **40,4 мс** |
| Marketplace LoRA | 0,7 | 0,6596 | 0,6974 | 66,2 мс |

LoRA обучалась на 221 438 автоматически размеченных примерах. Шум weak supervision ухудшил обобщающую способность, поэтому адаптер не используется в финальном решении.


In [ ]:
RUN_BASE_VS_LORA_COMPARISON = False


def tune_threshold_for_predictions(predictions):
    rows = []

    for threshold in np.arange(0.10, 0.71, 0.05):
        threshold = round(float(threshold), 2)

        metrics = calculate_all_metrics(
            predictions,
            threshold=threshold,
        )

        rows.append({
            "threshold": threshold,
            "span_f1": metrics["span_f1"],
            "span_precision": metrics["span_precision"],
            "span_recall": metrics["span_recall"],
            "surface_f1": metrics["surface_f1"],
            "attribute_f1": metrics["attribute_f1"],
            "hallucination_rate": metrics["hallucination_rate"],
        })

    table = (
        pd.DataFrame(rows)
        .sort_values(
            [
                "span_f1",
                "span_precision",
                "surface_f1",
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    return float(table.iloc[0]["threshold"]), table


def evaluate_gliner_variant(
    variant_name,
    use_adapter,
    shortlist_size=5,
):
    global extractor

    # Удаляем предыдущую модель.
    if "extractor" in globals():
        try:
            del extractor
        except Exception:
            pass

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    extractor = GLiNER2.from_pretrained(
        GLINER_BASE_MODEL,
        map_location="cuda" if torch.cuda.is_available() else "cpu",
    )

    if use_adapter:
        if not GLINER_ADAPTER_FINAL_DIR.exists():
            raise FileNotFoundError(
                f"Не найден LoRA-адаптер: "
                f"{GLINER_ADAPTER_FINAL_DIR}"
            )

        extractor.load_adapter(
            str(GLINER_ADAPTER_FINAL_DIR)
        )

    # Прямой smoke test — ошибка больше не будет скрыта.
    smoke_row = proxy_dev.iloc[0]

    smoke_prediction = predict_product_attributes(
        query=smoke_row["query_text"],
        gliner_threshold=0.10,
        category_top_k=3,
        shortlist_size=shortlist_size,
    )

    print(
        f"{variant_name}: smoke test, "
        f"найдено фактов:",
        len(smoke_prediction["flat_facts"]),
    )

    dev_result = run_golden_evaluation(
        proxy_dev,
        gliner_threshold=0.10,
        category_top_k=3,
        shortlist_size=shortlist_size,
        progress_description=f"{variant_name}: dev",
    )

    n_dev_errors = int(
        dev_result["execution_error"]
        .notna()
        .sum()
    )

    if n_dev_errors:
        print("\nПервые ошибки:")
        print(
            dev_result.loc[
                dev_result["execution_error"].notna(),
                "execution_error",
            ]
            .value_counts()
            .head(10)
        )

        raise RuntimeError(
            f"{variant_name}: ошибок на dev "
            f"{n_dev_errors}/{len(dev_result)}"
        )

    print(
        f"{variant_name}: фактов на dev:",
        int(dev_result["predicted_fact_count"].sum()),
    )

    best_threshold, threshold_table = (
        tune_threshold_for_predictions(dev_result)
    )

    print(
        f"{variant_name}: лучший threshold =",
        best_threshold,
    )

    test_result = run_golden_evaluation(
        proxy_test,
        gliner_threshold=best_threshold,
        category_top_k=3,
        shortlist_size=shortlist_size,
        progress_description=f"{variant_name}: test",
    )

    n_test_errors = int(
        test_result["execution_error"]
        .notna()
        .sum()
    )

    if n_test_errors:
        print(
            test_result.loc[
                test_result["execution_error"].notna(),
                "execution_error",
            ]
            .value_counts()
            .head(10)
        )

        raise RuntimeError(
            f"{variant_name}: ошибок на test "
            f"{n_test_errors}/{len(test_result)}"
        )

    metrics = calculate_all_metrics(test_result)

    result_row = {
        "variant": variant_name,
        "shortlist_size": shortlist_size,
        "best_threshold": best_threshold,
        **metrics,
    }

    return (
        result_row,
        threshold_table,
        dev_result,
        test_result,
    )

In [ ]:
if RUN_BASE_VS_LORA_COMPARISON:
    base_row, base_thresholds, base_dev, base_test = (
        evaluate_gliner_variant(
            variant_name="base",
            use_adapter=False,
            shortlist_size=5,
        )
    )

    lora_row, lora_thresholds, lora_dev, lora_test = (
        evaluate_gliner_variant(
            variant_name="marketplace_lora",
            use_adapter=True,
            shortlist_size=5,
        )
    )

    comparison = pd.DataFrame([base_row, lora_row])
    comparison.to_csv(GLINER_COMPARISON_PATH, index=False)

    display(
        comparison[
            [
                "variant",
                "shortlist_size",
                "best_threshold",
                "execution_errors",
                "span_precision",
                "span_recall",
                "span_f1",
                "surface_f1",
                "attribute_f1",
                "hallucination_rate",
                "latency_p50_ms",
                "latency_p95_ms",
            ]
        ]
    )
else:
    print("Base-vs-LoRA comparison отключён. Готовые метрики: results/base_vs_lora.csv")


## 15. Финальный вывод

Финальная конфигурация проекта:

```text
fastText category classifier
        ↓
dynamic shortlist-5
        ↓
base fastino/gliner2-multi-v1
```

Ключевые результаты:

- fastText validation: Accuracy@1 `0,7905`, Recall@3 `0,8845`;
- proxy-golden: Span F1 `0,7035`, Attribute F1 `0,7674`;
- Tesla T4: p50 `27,0 мс`, p95 `40,4 мс`;
- LoRA на 221 438 weak labels: Span F1 `0,6596`, поэтому отклонена.

